# ClaimCheck: Autonomous Analytical Review Agent for Business & Product Claims

**Authors:** Shivangi Kapoor & Shruthi Khurana  
**Course:** UC Davis MSBA · Agentic AI for Business · Summer 2026

## Project purpose

Business teams often move from dashboard results to executive recommendations too quickly. A stakeholder may say that a campaign “worked,” a product change “drove retention,” or a customer segment “should be targeted,” but the underlying evidence may not support the strength of that claim.

ClaimCheck is an agentic analytical review workflow designed to sit between analysis and decision-making. It converts a business claim into a testable hypothesis, profiles the available dataset, selects an appropriate analytical pathway, runs deterministic statistical checks, assigns an evidence-confidence tier, and generates an executive-ready recommendation with guardrails against unsupported causal language.

## What this notebook demonstrates

This notebook demonstrates the core analytical engine behind ClaimCheck:

1. **Claim-to-method routing:** converting a business claim and dataset structure into an appropriate analytical pathway.
2. **Deterministic validation:** using Python tools for data profiling, experimental validity checks, statistical testing, segment analysis, business impact, and audit logging.
3. **Evidence-tiered recommendations:** distinguishing between causal, suggestive, and descriptive evidence before generating executive-facing language.
4. **Exception-based review:** routing ambiguous, high-risk, or unsupported claims to human review instead of allowing automatic approval.
5. **Reusable design:** supporting multiple dataset types through a method registry rather than hard-coding a single analysis.

## Prototype scope

The current prototype focuses on common business decision patterns that can be handled through first-pass analytical review: campaign/product experiments, treatment-control comparisons, segment-level lift, profitability analysis, and cases where the data supports only descriptive or predictive interpretation.

The prototype implements selected high-frequency pathways, including binary treatment-control testing, continuous treatment-control testing, descriptive/predictive review, confidence-tiering, causal-language blocking, exception routing, and audit logging. Additional methods such as difference-in-differences, propensity score matching, interrupted time series, uplift modeling, and multi-armed bandits can be added later through the same governed method-registry design.


# Why ClaimCheck is Agentic

ClaimCheck is not simply a statistical testing notebook. A statistical notebook can run a test once the analyst already knows what to test. ClaimCheck instead performs a recurring analytical review workflow: it interprets a business claim, profiles the dataset, selects an appropriate method, runs deterministic validation tools, assigns an evidence-confidence tier, blocks unsupported causal language, routes exceptions to human reviewers, and generates an executive-facing recommendation.

The enterprise problem is that senior analysts and analytics managers repeatedly review business claims before they reach leadership: “Did this campaign work?”, “Should this segment be targeted?”, “Did this product change drive retention?”, or “Is this dashboard result decision-ready?” This review is important but repetitive, slow, and often skipped under deadline pressure.

ClaimCheck changes that workflow from manual review of every claim to autonomous first-pass review with escalation only for exceptions. The value is not replacing statistical packages or human judgment; the value is orchestrating claim interpretation, method selection, deterministic analysis, evidence-tiering, language control, and audit logging into one governed decision workflow.

## Why not just use a human analyst, statistical tool, or general LLM?

A human senior analyst provides judgment and context, but manual review of every claim can become slow, expensive, and inconsistent. Statistical tools can calculate tests accurately, but they usually do not interpret messy business claims, decide whether causal language is allowed, or produce an auditable decision workflow. General-purpose LLMs such as GPT or Claude can explain and draft recommendations, but by themselves they are not deterministic, method-governed, or audit-safe for statistical claims.

ClaimCheck combines these strengths while controlling their risks: Python performs the calculations, LLM agents handle interpretation and communication, and the workflow uses confidence tiers, exception routing, and audit logs to preserve human oversight where it matters.

# System Architecture

ClaimCheck uses a minimal multi-agent design supported by deterministic analytical tools. The system is intentionally not a single chatbot and not a fully manual statistical notebook. Each component has a separate responsibility so that claim interpretation, data validation, method selection, statistical testing, risk review, and executive communication remain auditable.

## ClaimCheck Crew

| Component | Type | Corporate role | Responsibility |
|---|---|---|---|
| Claim Framer | LLM agent | Senior Business Analyst | Converts a messy business claim into a structured, testable hypothesis. |
| Data Profiler | Deterministic Python tool | Analytics Engineer | Checks dataset schema, missing values, sample size, candidate treatment/control variables, and dataset hash. |
| Method Scout | LLM + rules agent | Decision Scientist | Selects the appropriate analytical pathway from the approved method registry. |
| Stats Engine | Deterministic Python tool | Data Scientist / Statistician | Runs approved statistical tests and returns structured numerical outputs. |
| Validity Guard | LLM + rules agent | Analytics Governance Reviewer | Assigns evidence tier, flags overclaiming risk, and decides whether human review is required. |
| Brief Builder | LLM agent | Executive Communication Lead | Generates an executive-ready recommendation constrained by the evidence tier. |

## Design principle

ClaimCheck does not delegate numerical calculations to an LLM. Statistical calculations are handled by deterministic Python tools using trusted analytical libraries. LLM agents are used only where language interpretation, method reasoning, risk explanation, and executive communication are needed.

## Human review gates

ClaimCheck is designed for autonomous first-pass review, not full replacement of human judgment. Most routine claims can be processed automatically, but the workflow escalates to human review when:

- the analytical method is ambiguous;
- required data is missing or low quality;
- observational data is being used for a causal claim;
- segment results conflict with the overall result;
- the requested recommendation has high business impact;
- the confidence tier is lower than the strength of the claim.

This keeps the workflow useful for enterprise settings: senior analysts do not need to review every routine claim, but they remain involved when the evidence is uncertain or the decision risk is high.

## Handoff model

Each stage passes structured JSON-like outputs to the next stage rather than long free-text summaries. This reduces truncation risk and keeps the workflow auditable.

```text
Business claim
   ↓
Claim Framer
   ↓
Structured hypothesis
   ↓
Data Profiler
   ↓
Dataset profile + data quality flags
   ↓
Method Scout
   ↓
Approved analytical pathway
   ↓
Stats Engine
   ↓
Statistical result package
   ↓
Validity Guard
   ↓
Confidence tier + exception decision
   ↓
Brief Builder
   ↓
Executive recommendation + audit log


## 1. Setup and Core Libraries

This cell loads the core libraries used by ClaimCheck. The notebook separates deterministic analysis from LLM-style reasoning: Python libraries handle data profiling, statistical testing, structured outputs, and audit logging. Later, CrewAI agents can orchestrate these tools, but the numerical calculations remain reproducible.


In [1]:
# ============================================================
# ClaimCheck: Setup and Core Libraries
# ============================================================

import pandas as pd
import numpy as np
import json
import hashlib
import datetime as dt

from typing import Dict, List, Any, Optional
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

pd.set_option("display.max_columns", 100)

print("ClaimCheck setup complete: data handling, statistical testing, structured outputs, and audit logging libraries are loaded.")

ClaimCheck setup complete: data handling, statistical testing, structured outputs, and audit logging libraries are loaded.


## 2. Product Configuration and Method Registry

A real enterprise version of ClaimCheck could receive claims from Slack, Google Docs, BI dashboards, spreadsheet uploads, or another workflow agent. In this prototype, the claim, decision context, and dataset mapping are passed through a configuration object.

This design keeps the engine reusable across datasets. The same ClaimCheck workflow can review a campaign experiment, product test, churn driver analysis, pre/post intervention, or descriptive dashboard claim as long as the dataset structure and business claim are clearly specified.

The method registry is the scalable part of the design. ClaimCheck does not try to place every statistical test inside an LLM prompt. Instead, it routes claims to approved analytical method families. Each registry entry defines when the method applies, what inputs are required, whether causal language may be allowed, and what implementation path should be used..


In [2]:
# ============================================================
# ClaimCheck: Configuration Template and Method Registry
# ============================================================

# Example configuration template.
# Each dataset/use case will create its own config using this structure.
example_config = {
    "dataset_name": "Example Dataset",
    "claim": "The intervention improved the business outcome.",
    "decision_context": "Product, campaign, pricing, retention, or operational decision",

    # Core analytical fields
    "outcome_col": None,          # Example: purchase, churn, revenue, satisfaction_score
    "treatment_col": None,        # Example: test, treatment, exposed, variant
    "group_col": None,            # Example: plan_type, region, campaign_variant
    "time_col": None,             # Example: date, week, month
    "unit_id_col": None,          # Example: customer_id, user_id, account_id

    # Optional analytical fields
    "covariates": [],             # Example: income, tenure, region, prior_spend
    "segment_cols": [],           # Example: gender, gamer, plan_type, customer_segment

    # Business impact assumptions
    "business_value_per_success": None,
    "intervention_cost_per_success": None,

    # Claim and risk settings
    "requested_claim_strength": "causal",   # descriptive, suggestive, causal
    "alpha": 0.05
}


METHOD_REGISTRY = {
    "two_proportion_test": {
        "business_case": "Conversion, purchase, churn, click, signup, renewal, or other binary outcome",
        "use_when": "Two independent groups are available and the outcome is binary",
        "required_inputs": ["treatment_col", "outcome_col"],
        "example_question": "Did the campaign increase purchase rate?",
        "implementation": "statsmodels.stats.proportion.proportions_ztest",
        "claim_language": "Causal language may be allowed only if assignment is experimental and validity checks pass",
        "prototype_status": "implemented"
    },

    "two_sample_t_test": {
        "business_case": "Revenue, spend, time-on-site, satisfaction score, usage, or other continuous outcome",
        "use_when": "Two independent groups are available and the outcome is continuous",
        "required_inputs": ["treatment_col", "outcome_col"],
        "example_question": "Did the new onboarding flow increase average usage?",
        "implementation": "scipy.stats.ttest_ind",
        "claim_language": "Causal language may be allowed only if assignment is experimental and validity checks pass",
        "prototype_status": "implemented"
    },

    "anova_multiple_groups": {
        "business_case": "Multiple campaign variants, pricing tiers, product versions, regions, or customer groups",
        "use_when": "Three or more groups are compared on a continuous outcome",
        "required_inputs": ["group_col", "outcome_col"],
        "example_question": "Which of several pricing variants performed best?",
        "implementation": "scipy/statsmodels ANOVA with post-hoc testing in future version",
        "claim_language": "Requires multiple-comparison control before strong claims",
        "prototype_status": "recommended / future implementation"
    },

    "pre_post_analysis": {
        "business_case": "Before/after policy, product, pricing, or process change",
        "use_when": "Outcome is observed before and after an intervention, but no separate control group is available",
        "required_inputs": ["time_col", "outcome_col"],
        "example_question": "Did churn decrease after the new retention program launched?",
        "implementation": "paired t-test or interrupted time-series in future version",
        "claim_language": "Usually suggestive, not causal, unless stronger design assumptions are justified",
        "prototype_status": "routing only"
    },

    "difference_in_differences": {
        "business_case": "Rollout to one market, team, region, or customer group while another remains untreated",
        "use_when": "Treatment and control groups are observed before and after the intervention",
        "required_inputs": ["treatment_col", "time_col", "outcome_col"],
        "example_question": "Did launching the new feature in one region improve retention relative to other regions?",
        "implementation": "statsmodels regression with treatment, post, and interaction terms in future version",
        "claim_language": "May support causal language only if parallel-trend assumptions are credible",
        "prototype_status": "routing only"
    },

    "regression_adjusted_analysis": {
        "business_case": "Observational business data where analysts want to adjust for covariates",
        "use_when": "No randomized assignment exists, but outcome and predictors are available",
        "required_inputs": ["outcome_col", "covariates"],
        "example_question": "Is discount exposure associated with higher renewal after controlling for customer tenure?",
        "implementation": "statsmodels OLS or logistic regression in future version",
        "claim_language": "Suggestive only unless supported by a stronger causal design",
        "prototype_status": "routing only"
    },

    "predictive_driver_review": {
        "business_case": "Churn scores, propensity models, feature importances, lead scores, or risk drivers",
        "use_when": "Dataset contains predictions, scores, or driver importances rather than an experimental comparison",
        "required_inputs": ["available columns"],
        "example_question": "Which factors are associated with higher churn risk?",
        "implementation": "pandas profiling and model-output review",
        "claim_language": "Descriptive or predictive language only; causal verbs blocked",
        "prototype_status": "implemented as review route"
    },

    "descriptive_only": {
        "business_case": "Dashboard summaries, KPI movement, cohort summaries, or datasets with no valid comparison group",
        "use_when": "The dataset cannot support a causal or statistical comparison claim",
        "required_inputs": ["available columns"],
        "example_question": "What changed in this KPI last month?",
        "implementation": "pandas descriptive profiling",
        "claim_language": "Descriptive language only",
        "prototype_status": "implemented as stop route"
    }
}

print(f"ClaimCheck method registry loaded with {len(METHOD_REGISTRY)} method families.")

ClaimCheck method registry loaded with 8 method families.


# 3. Core Deterministic Tools and Guardrails

These are the tools that ClaimCheck agents would call during the workflow. They are deliberately written as generic Python functions so they can be reused across datasets, claims, and future CrewAI agents.

The design principle is simple: agents handle interpretation, routing context, and executive communication, while deterministic tools handle calculation, validation, enforcement, and auditability.

This separation is important because statistical testing, data profiling, business-impact calculation, and language enforcement must be reproducible. An LLM can help frame a claim or explain an output, but it should not be trusted to calculate p-values, infer dataset structure, or decide unsupported causal language without deterministic checks.

## 3.1 Data Profiler Tool

The Data Profiler is the first deterministic tool in ClaimCheck. It inspects the dataset before any statistical method is selected.

This tool checks dataset size, column names, missing values, likely binary columns, likely numeric columns, likely categorical/group columns, configured-column availability, treatment/control structure, and a dataset hash for auditability.

This step prevents the rest of the workflow from guessing what the data contains. The Method Scout should choose an analytical pathway only after this profile is created.

In [3]:
# ============================================================
# 3.1 Data Profiler Tool
# ============================================================

def dataset_hash(df: pd.DataFrame) -> str:
    """
    Create a short reproducibility hash from dataframe values and column names.
    This links each ClaimCheck recommendation to a specific dataset version.
    """
    safe_df = df.copy()

    # Convert object columns to string so mixed text/object columns hash consistently.
    for col in safe_df.columns:
        if safe_df[col].dtype == "object":
            safe_df[col] = safe_df[col].astype(str)

    hashed_values = pd.util.hash_pandas_object(
        safe_df.fillna("<NA>"),
        index=True
    ).values

    digest = hashlib.sha256(
        hashed_values.tobytes() + "|".join(safe_df.columns).encode()
    ).hexdigest()

    return digest[:16]


def is_binary_series(series: pd.Series) -> bool:
    """
    Detect whether a column behaves like a binary 0/1 or True/False variable.
    Binary outcomes usually need a two-proportion test.
    """
    values = set(pd.Series(series.dropna().unique()).tolist())
    return len(values) <= 2 and values.issubset({0, 1, 0.0, 1.0, True, False})


def detect_column_roles(df: pd.DataFrame) -> Dict[str, List[str]]:
    """
    Detect likely column roles using transparent first-pass rules.
    This does not replace analyst judgment, but supports method routing.
    """
    binary_cols = []
    numeric_cols = []
    categorical_cols = []
    datetime_cols = []

    for col in df.columns:
        series = df[col].dropna()

        if series.empty:
            continue

        unique_count = series.nunique()

        if pd.api.types.is_datetime64_any_dtype(series):
            datetime_cols.append(col)

        elif pd.api.types.is_numeric_dtype(series):
            numeric_cols.append(col)

            if is_binary_series(series):
                binary_cols.append(col)

            # Numeric columns with few unique values may also behave like groups.
            if unique_count <= 20:
                categorical_cols.append(col)

        else:
            # Text columns with limited unique values are candidate group columns.
            if unique_count <= 50:
                categorical_cols.append(col)

    return {
        "binary_cols": binary_cols,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "datetime_cols": datetime_cols
    }


def validate_config_columns(df: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Check whether columns named in the config exist in the dataset.
    """
    single_column_fields = [
        "outcome_col",
        "treatment_col",
        "group_col",
        "time_col",
        "unit_id_col"
    ]

    missing_configured_columns = []

    for field in single_column_fields:
        col = config.get(field)
        if col is not None and col not in df.columns:
            missing_configured_columns.append({
                "config_field": field,
                "column_name": col
            })

    for list_field in ["covariates", "segment_cols"]:
        for col in config.get(list_field, []):
            if col not in df.columns:
                missing_configured_columns.append({
                    "config_field": list_field,
                    "column_name": col
                })

    return {
        "all_configured_columns_found": len(missing_configured_columns) == 0,
        "missing_configured_columns": missing_configured_columns
    }


def profile_dataset(df: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Main Data Profiler function.
    Returns a structured dataset profile for downstream method selection.
    """
    role_detection = detect_column_roles(df)
    config_validation = validate_config_columns(df, config)

    missing_values = {
        col: int(df[col].isna().sum())
        for col in df.columns
        if int(df[col].isna().sum()) > 0
    }

    treatment_values = None
    has_two_group_treatment = False

    treatment_col = config.get("treatment_col")
    if treatment_col and treatment_col in df.columns:
        treatment_values = sorted(pd.Series(df[treatment_col].dropna().unique()).tolist())
        has_two_group_treatment = len(treatment_values) == 2

    outcome_type = None
    outcome_col = config.get("outcome_col")
    if outcome_col and outcome_col in df.columns:
        if is_binary_series(df[outcome_col]):
            outcome_type = "binary"
        elif pd.api.types.is_numeric_dtype(df[outcome_col]):
            outcome_type = "continuous_or_numeric"
        else:
            outcome_type = "categorical_or_text"

    profile = {
        "dataset_name": config.get("dataset_name"),
        "dataset_hash": dataset_hash(df),
        "n_rows": int(df.shape[0]),
        "n_columns": int(df.shape[1]),
        "columns": list(df.columns),
        "column_roles": role_detection,
        "missing_values": missing_values,
        "config_validation": config_validation,
        "treatment_values": treatment_values,
        "has_two_group_treatment": has_two_group_treatment,
        "outcome_type": outcome_type,
        "candidate_segments": [
            col for col in config.get("segment_cols", [])
            if col in df.columns
        ],
        "candidate_covariates": [
            col for col in config.get("covariates", [])
            if col in df.columns
        ],
        "flags": []
    }

    if not config_validation["all_configured_columns_found"]:
        profile["flags"].append("One or more configured columns are missing from the dataset.")

    if outcome_col is None or outcome_col not in df.columns:
        profile["flags"].append("Outcome column is missing or not mapped.")

    if treatment_col and treatment_col in df.columns and not has_two_group_treatment:
        profile["flags"].append("Treatment column does not contain exactly two groups.")

    if len(df) < 200:
        profile["flags"].append("Small sample size: human review recommended.")

    return profile


def print_dataset_profile(profile: Dict[str, Any]) -> None:
    """
    Display the dataset profile in a readable format for the notebook.
    """
    print("DATASET PROFILE")
    print("=" * 70)
    print(f"Dataset: {profile['dataset_name']}")
    print(f"Dataset hash: {profile['dataset_hash']}")
    print(f"Rows: {profile['n_rows']:,}")
    print(f"Columns: {profile['n_columns']}")
    print(f"Outcome type: {profile['outcome_type']}")
    print(f"Two-group treatment structure: {profile['has_two_group_treatment']}")

    print("\nDetected column roles:")
    print(json.dumps(profile["column_roles"], indent=2))

    print("\nConfig validation:")
    print(json.dumps(profile["config_validation"], indent=2))

    if profile["missing_values"]:
        print("\nMissing values:")
        print(json.dumps(profile["missing_values"], indent=2))
    else:
        print("\nMissing values: none detected")

    if profile["flags"]:
        print("\nProfile flags:")
        for flag in profile["flags"]:
            print(f"- {flag}")
    else:
        print("\nProfile flags: none")

print("Data Profiler Tool defined.")

Data Profiler Tool defined.


## 3.2 Method Selection Engine

The Method Selection Engine acts as the deterministic backbone of the Method Scout agent. It uses the dataset profile, configuration object, and approved method registry to decide which analytical pathway is appropriate.

This step is important because ClaimCheck should not blindly run an A/B test on every dataset. A product manager or senior analyst may ask different kinds of questions: whether a campaign increased conversion, whether a product change affected average usage, whether multiple variants differ, whether a pre/post change is suggestive, or whether a churn-driver dataset only supports predictive interpretation.

The method selector returns a structured decision containing:

- selected method family;
- whether the method can run automatically in this prototype;
- whether human review is required;
- whether causal language may be allowed;
- the reason for the routing decision.

This keeps method selection auditable and prevents unsupported claims from moving directly into executive recommendations.

In [4]:
# ============================================================
# 3.2: Method Selection Engine
# ============================================================

def infer_claim_intent(config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Infer the requested claim strength and broad business intent from the configuration.
    This is a simple rule-based version of what the Claim Framer agent would later help structure.
    """
    claim_text = str(config.get("claim", "")).lower()
    requested_strength = config.get("requested_claim_strength", "causal").lower()

    causal_terms = ["caused", "drove", "increased", "reduced", "improved", "lifted", "impact", "effect"]
    predictive_terms = ["predict", "propensity", "risk", "likelihood", "score", "driver"]
    rollout_terms = ["roll out", "launch", "target", "recommend", "continue", "scale"]

    return {
        "requested_claim_strength": requested_strength,
        "contains_causal_language": any(term in claim_text for term in causal_terms),
        "contains_predictive_language": any(term in claim_text for term in predictive_terms),
        "contains_decision_language": any(term in claim_text for term in rollout_terms)
    }


def select_method(profile: Dict[str, Any], config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Select the most appropriate approved method family from the method registry.
    This function does not run the statistical test. It only routes the claim to the right pathway.
    """
    claim_intent = infer_claim_intent(config)

    outcome_col = config.get("outcome_col")
    treatment_col = config.get("treatment_col")
    group_col = config.get("group_col")
    time_col = config.get("time_col")
    covariates = config.get("covariates", [])

    columns = profile.get("columns", [])
    outcome_type = profile.get("outcome_type")
    has_two_group_treatment = profile.get("has_two_group_treatment", False)
    config_valid = profile.get("config_validation", {}).get("all_configured_columns_found", False)

    # 1. Missing or invalid configured columns
    if not config_valid or outcome_col is None or outcome_col not in columns:
        return {
            "method_id": "descriptive_only",
            "method_status": METHOD_REGISTRY["descriptive_only"]["prototype_status"],
            "auto_runnable": False,
            "human_review_required": True,
            "causal_language_allowed": False,
            "reason": "Required configured fields are missing or the outcome column is not mapped.",
            "claim_intent": claim_intent
        }

    # 2. Clean two-group treatment/control pathway
    if treatment_col and treatment_col in columns and has_two_group_treatment:
        if outcome_type == "binary":
            return {
                "method_id": "two_proportion_test",
                "method_status": METHOD_REGISTRY["two_proportion_test"]["prototype_status"],
                "auto_runnable": True,
                "human_review_required": False,
                "causal_language_allowed": True,
                "reason": "Two-group treatment/control structure detected with a binary outcome.",
                "claim_intent": claim_intent
            }

        if outcome_type == "continuous_or_numeric":
            return {
                "method_id": "two_sample_t_test",
                "method_status": METHOD_REGISTRY["two_sample_t_test"]["prototype_status"],
                "auto_runnable": True,
                "human_review_required": False,
                "causal_language_allowed": True,
                "reason": "Two-group treatment/control structure detected with a continuous or numeric outcome.",
                "claim_intent": claim_intent
            }

    # 3. Multiple-group comparison pathway
    if group_col and group_col in columns:
        return {
            "method_id": "anova_multiple_groups",
            "method_status": METHOD_REGISTRY["anova_multiple_groups"]["prototype_status"],
            "auto_runnable": False,
            "human_review_required": True,
            "causal_language_allowed": False,
            "reason": "Multiple-group comparison detected. ANOVA or post-hoc testing plan is needed before strong claims.",
            "claim_intent": claim_intent
        }

    # 4. Difference-in-differences candidate pathway
    if treatment_col and treatment_col in columns and time_col and time_col in columns:
        return {
            "method_id": "difference_in_differences",
            "method_status": METHOD_REGISTRY["difference_in_differences"]["prototype_status"],
            "auto_runnable": False,
            "human_review_required": True,
            "causal_language_allowed": False,
            "reason": "Treatment and time fields exist, suggesting a possible difference-in-differences design. Prototype routes this for human review.",
            "claim_intent": claim_intent
        }

    # 5. Pre/post pathway
    if time_col and time_col in columns:
        return {
            "method_id": "pre_post_analysis",
            "method_status": METHOD_REGISTRY["pre_post_analysis"]["prototype_status"],
            "auto_runnable": False,
            "human_review_required": True,
            "causal_language_allowed": False,
            "reason": "Time field exists but no valid control group is mapped. Pre/post analysis may be suggestive, not causal.",
            "claim_intent": claim_intent
        }

    # 6. Observational adjusted analysis pathway
    if outcome_col in columns and len(covariates) > 0:
        available_covariates = [c for c in covariates if c in columns]
        if len(available_covariates) > 0:
            return {
                "method_id": "regression_adjusted_analysis",
                "method_status": METHOD_REGISTRY["regression_adjusted_analysis"]["prototype_status"],
                "auto_runnable": False,
                "human_review_required": True,
                "causal_language_allowed": False,
                "reason": "Outcome and covariates are available, but no randomized comparison is mapped. Adjusted observational analysis may be suggestive only.",
                "claim_intent": claim_intent
            }

    # 7. Predictive / driver review pathway
    if claim_intent["contains_predictive_language"]:
        return {
            "method_id": "predictive_driver_review",
            "method_status": METHOD_REGISTRY["predictive_driver_review"]["prototype_status"],
            "auto_runnable": True,
            "human_review_required": True,
            "causal_language_allowed": False,
            "reason": "Claim appears predictive or driver-oriented rather than experimental. Causal language should be blocked.",
            "claim_intent": claim_intent
        }

    # 8. Default safe route
    return {
        "method_id": "descriptive_only",
        "method_status": METHOD_REGISTRY["descriptive_only"]["prototype_status"],
        "auto_runnable": True,
        "human_review_required": True,
        "causal_language_allowed": False,
        "reason": "No valid experimental, multi-group, pre/post, or adjusted analysis pathway was detected. Descriptive review only.",
        "claim_intent": claim_intent
    }


def print_method_decision(method_decision: Dict[str, Any]) -> None:
    """
    Display method routing decision in a readable format.
    """
    method_id = method_decision["method_id"]
    registry_entry = METHOD_REGISTRY.get(method_id, {})

    print("METHOD SELECTION DECISION")
    print("=" * 70)
    print(f"Selected method family: {method_id}")
    print(f"Prototype status: {method_decision['method_status']}")
    print(f"Auto-runnable in prototype: {method_decision['auto_runnable']}")
    print(f"Human review required: {method_decision['human_review_required']}")
    print(f"Causal language allowed at this stage: {method_decision['causal_language_allowed']}")
    print(f"Reason: {method_decision['reason']}")

    if registry_entry:
        print("\nRegistry metadata:")
        print(f"Business case: {registry_entry.get('business_case')}")
        print(f"Use when: {registry_entry.get('use_when')}")
        print(f"Implementation: {registry_entry.get('implementation')}")
        print(f"Claim language rule: {registry_entry.get('claim_language')}")

    print("\nClaim intent:")
    print(json.dumps(method_decision["claim_intent"], indent=2))


print("Method Selection Engine defined.")

Method Selection Engine defined.


## 3.3 Statistical Validation and Safe Routing Tools

The Stats Engine is the deterministic analytical layer of ClaimCheck. It executes statistical methods only when the selected method is implemented and the required data structure is available.

The Method Registry recognizes several common business-analysis pathways: treatment-control experiments, multiple-group comparisons, pre/post analysis, difference-in-differences, observational adjusted analysis, predictive/driver review, and descriptive-only review. However, this prototype intentionally implements only the highest-frequency first-pass pathways and safely routes unsupported methods to human review.

In this version, the Stats Engine implements:

1. **Two-proportion testing** for binary outcomes such as purchase, churn, click, signup, or renewal.
2. **Two-sample t-testing** for continuous outcomes such as revenue, spend, usage, satisfaction, or time-on-site.
3. **Predictive/driver review routing** for model scores, feature importances, churn-risk outputs, or datasets that do not support causal claims.
4. **Descriptive-only routing** for dashboard-style claims, KPI summaries, or datasets without a valid comparison group.

Advanced methods such as ANOVA post-hoc testing, regression adjustment, difference-in-differences, interrupted time series, propensity score matching, uplift modeling, and anomaly detection are recognized by the registry but not fully executed in this prototype. This is intentional: ClaimCheck should not run a method just because it sounds relevant. It should execute only approved methods and escalate unsupported cases.

In [5]:
# ============================================================
# 3.3: Statistical Validation and Safe Routing Tools
# ============================================================

def get_two_groups(df: pd.DataFrame, treatment_col: str) -> Dict[str, Any]:
    """
    Extract exactly two groups from a treatment/control column.
    Assumes the Data Profiler has already checked that two groups exist.
    """
    values = sorted(pd.Series(df[treatment_col].dropna().unique()).tolist())

    if len(values) != 2:
        raise ValueError(
            f"Expected exactly two groups in '{treatment_col}', found {len(values)}: {values}"
        )

    control_value = values[0]
    treatment_value = values[1]

    return {
        "control_value": control_value,
        "treatment_value": treatment_value,
        "control_df": df[df[treatment_col] == control_value].copy(),
        "treatment_df": df[df[treatment_col] == treatment_value].copy()
    }


def check_group_balance(df: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Check treatment/control balance on configured covariates.
    Binary covariates use a two-proportion z-test.
    Numeric covariates use Welch's t-test.
    """
    treatment_col = config.get("treatment_col")
    covariates = config.get("covariates", [])
    alpha = config.get("alpha", 0.05)

    if treatment_col is None or treatment_col not in df.columns:
        return {
            "available": False,
            "balanced": None,
            "reason": "No treatment column is mapped or available.",
            "checks": {}
        }

    try:
        groups = get_two_groups(df, treatment_col)
    except ValueError as error:
        return {
            "available": False,
            "balanced": None,
            "reason": str(error),
            "checks": {}
        }

    control_df = groups["control_df"]
    treatment_df = groups["treatment_df"]
    checks = {}

    for col in covariates:
        if col not in df.columns:
            continue

        if is_binary_series(df[col]):
            control_successes = int(control_df[col].sum())
            treatment_successes = int(treatment_df[col].sum())
            control_n = int(control_df[col].notna().sum())
            treatment_n = int(treatment_df[col].notna().sum())

            stat, p_value = proportions_ztest(
                count=[treatment_successes, control_successes],
                nobs=[treatment_n, control_n]
            )

            test_name = "two_proportion_z_test"

        elif pd.api.types.is_numeric_dtype(df[col]):
            stat, p_value = stats.ttest_ind(
                treatment_df[col].dropna(),
                control_df[col].dropna(),
                equal_var=False
            )

            test_name = "welch_t_test"

        else:
            continue

        checks[col] = {
            "test": test_name,
            "control_mean": float(control_df[col].mean()),
            "treatment_mean": float(treatment_df[col].mean()),
            "difference": float(treatment_df[col].mean() - control_df[col].mean()),
            "p_value": float(p_value),
            "balanced_at_alpha": bool(p_value >= alpha)
        }

    if len(checks) == 0:
        balanced = None
        reason = "No valid covariates were available for balance checking."
    else:
        balanced = all(item["balanced_at_alpha"] for item in checks.values())
        reason = "Balance checks completed."

    return {
        "available": True,
        "balanced": balanced,
        "reason": reason,
        "checks": checks,
        "control_value": groups["control_value"],
        "treatment_value": groups["treatment_value"]
    }


def run_two_proportion_test(df: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Run a two-proportion z-test for a binary outcome.
    Used for purchase, churn, click, signup, renewal, or conversion outcomes.
    """
    treatment_col = config.get("treatment_col")
    outcome_col = config.get("outcome_col")
    alpha = config.get("alpha", 0.05)

    groups = get_two_groups(df, treatment_col)
    control_df = groups["control_df"]
    treatment_df = groups["treatment_df"]

    control_successes = int(control_df[outcome_col].sum())
    treatment_successes = int(treatment_df[outcome_col].sum())
    control_n = int(control_df[outcome_col].notna().sum())
    treatment_n = int(treatment_df[outcome_col].notna().sum())

    control_rate = control_successes / control_n
    treatment_rate = treatment_successes / treatment_n
    effect = treatment_rate - control_rate

    stat, p_value = proportions_ztest(
        count=[treatment_successes, control_successes],
        nobs=[treatment_n, control_n]
    )

    standard_error = np.sqrt(
        (treatment_rate * (1 - treatment_rate) / treatment_n) +
        (control_rate * (1 - control_rate) / control_n)
    )

    ci_low = effect - 1.96 * standard_error
    ci_high = effect + 1.96 * standard_error

    return {
        "method": "two_proportion_test",
        "outcome_col": outcome_col,
        "treatment_col": treatment_col,
        "control_value": groups["control_value"],
        "treatment_value": groups["treatment_value"],
        "control_n": control_n,
        "treatment_n": treatment_n,
        "control_successes": control_successes,
        "treatment_successes": treatment_successes,
        "control_rate": float(control_rate),
        "treatment_rate": float(treatment_rate),
        "effect": float(effect),
        "effect_pp": float(effect * 100),
        "p_value": float(p_value),
        "ci_95_low": float(ci_low),
        "ci_95_high": float(ci_high),
        "ci_95_low_pp": float(ci_low * 100),
        "ci_95_high_pp": float(ci_high * 100),
        "statistically_significant": bool(p_value < alpha)
    }


def run_two_sample_t_test(df: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Run Welch's two-sample t-test for a continuous or numeric outcome.
    Used for revenue, spend, usage, satisfaction, or time-on-site outcomes.
    """
    treatment_col = config.get("treatment_col")
    outcome_col = config.get("outcome_col")
    alpha = config.get("alpha", 0.05)

    groups = get_two_groups(df, treatment_col)
    control_values = groups["control_df"][outcome_col].dropna()
    treatment_values = groups["treatment_df"][outcome_col].dropna()

    stat, p_value = stats.ttest_ind(
        treatment_values,
        control_values,
        equal_var=False
    )

    control_mean = control_values.mean()
    treatment_mean = treatment_values.mean()
    effect = treatment_mean - control_mean

    return {
        "method": "two_sample_t_test",
        "outcome_col": outcome_col,
        "treatment_col": treatment_col,
        "control_value": groups["control_value"],
        "treatment_value": groups["treatment_value"],
        "control_n": int(len(control_values)),
        "treatment_n": int(len(treatment_values)),
        "control_mean": float(control_mean),
        "treatment_mean": float(treatment_mean),
        "effect": float(effect),
        "p_value": float(p_value),
        "statistically_significant": bool(p_value < alpha)
    }


def run_statistical_validation(
    df: pd.DataFrame,
    config: Dict[str, Any],
    method_decision: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Run the selected statistical validation pathway.
    Unsupported methods are safely routed instead of being forced.
    """
    method_id = method_decision.get("method_id")

    validation_package = {
        "method_id": method_id,
        "method_status": method_decision.get("method_status"),
        "auto_runnable": method_decision.get("auto_runnable"),
        "statistical_result": None,
        "balance_result": None,
        "ran_successfully": False,
        "notes": []
    }

    if method_id in ["two_proportion_test", "two_sample_t_test"]:
        validation_package["balance_result"] = check_group_balance(df, config)

        if method_id == "two_proportion_test":
            validation_package["statistical_result"] = run_two_proportion_test(df, config)
        else:
            validation_package["statistical_result"] = run_two_sample_t_test(df, config)

        validation_package["ran_successfully"] = True
        validation_package["notes"].append("Implemented statistical pathway ran successfully.")
        return validation_package

    if method_id == "predictive_driver_review":
        validation_package["ran_successfully"] = True
        validation_package["notes"].append(
            "Predictive/driver review route selected. No causal statistical test is run in this prototype."
        )
        return validation_package

    if method_id == "descriptive_only":
        validation_package["ran_successfully"] = True
        validation_package["notes"].append(
            "Descriptive-only route selected. No inferential or causal statistical test is run."
        )
        return validation_package

    validation_package["notes"].append(
        f"The selected method '{method_id}' is recognized by the registry but not implemented in this prototype."
    )
    validation_package["notes"].append(
        "ClaimCheck routes this case to human review rather than running an unsupported test."
    )

    return validation_package


def print_validation_package(validation_package: Dict[str, Any]) -> None:
    """
    Display statistical validation results in a readable format.
    """
    print("STATISTICAL VALIDATION PACKAGE")
    print("=" * 70)
    print(f"Method: {validation_package['method_id']}")
    print(f"Ran successfully: {validation_package['ran_successfully']}")

    print("\nNotes:")
    for note in validation_package["notes"]:
        print(f"- {note}")

    if validation_package["balance_result"] is not None:
        print("\nBalance result:")
        print(json.dumps(validation_package["balance_result"], indent=2))

    if validation_package["statistical_result"] is not None:
        print("\nStatistical result:")
        print(json.dumps(validation_package["statistical_result"], indent=2))


print("Statistical Validation and Safe Routing Tools defined.")

Statistical Validation and Safe Routing Tools defined.


## 3.4 Segment and Business Impact Tools

Business recommendations often require more than an average treatment effect. A campaign, product feature, or pricing change may appear positive overall but perform very differently across customer segments. A senior analyst or product manager also needs to know whether the effect is commercially meaningful, not only statistically significant.

This section adds two deterministic tool layers:

1. **Segment analysis:** estimates treatment effects within configured customer or product segments.
2. **Business impact analysis:** uses value and cost assumptions to evaluate whether a rollout appears financially supported.

These tools help ClaimCheck distinguish between broad rollout, targeted rollout, and human-review cases.

In [6]:
# ============================================================
# ClaimCheck 3.4: Segment and Business Impact Tools
# ============================================================

def run_segment_effects(df: pd.DataFrame, config: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Run treatment-effect analysis within configured segment columns.
    This helps identify whether the overall effect hides meaningful segment differences.
    """
    segment_cols = config.get("segment_cols", [])
    treatment_col = config.get("treatment_col")
    outcome_col = config.get("outcome_col")

    if not segment_cols or treatment_col is None or outcome_col is None:
        return []

    results = []

    for segment_col in segment_cols:
        if segment_col not in df.columns:
            continue

        segment_values = sorted(pd.Series(df[segment_col].dropna().unique()).tolist())

        for segment_value in segment_values:
            subset = df[df[segment_col] == segment_value].copy()

            # Only run if the segment contains both treatment and control groups.
            if treatment_col not in subset.columns or subset[treatment_col].nunique() != 2:
                continue

            # Avoid very tiny segment calculations.
            if len(subset) < 50:
                continue

            try:
                if is_binary_series(subset[outcome_col]):
                    result = run_two_proportion_test(subset, config)
                elif pd.api.types.is_numeric_dtype(subset[outcome_col]):
                    result = run_two_sample_t_test(subset, config)
                else:
                    continue

                result["segment_col"] = segment_col
                result["segment_value"] = segment_value
                result["segment_n"] = int(len(subset))
                results.append(result)

            except Exception as error:
                results.append({
                    "segment_col": segment_col,
                    "segment_value": segment_value,
                    "segment_n": int(len(subset)),
                    "error": str(error)
                })

    # Optional two-way segment interaction for the first two segment columns.
    if len(segment_cols) >= 2:
        first_segment = segment_cols[0]
        second_segment = segment_cols[1]

        if first_segment in df.columns and second_segment in df.columns:
            combinations = (
                df[[first_segment, second_segment]]
                .dropna()
                .drop_duplicates()
                .itertuples(index=False, name=None)
            )

            for combo in sorted(combinations):
                subset = df[
                    (df[first_segment] == combo[0]) &
                    (df[second_segment] == combo[1])
                ].copy()

                if treatment_col not in subset.columns or subset[treatment_col].nunique() != 2:
                    continue

                if len(subset) < 50:
                    continue

                try:
                    if is_binary_series(subset[outcome_col]):
                        result = run_two_proportion_test(subset, config)
                    elif pd.api.types.is_numeric_dtype(subset[outcome_col]):
                        result = run_two_sample_t_test(subset, config)
                    else:
                        continue

                    result["segment_col"] = f"{first_segment}+{second_segment}"
                    result["segment_value"] = f"{combo[0]}+{combo[1]}"
                    result["segment_n"] = int(len(subset))
                    results.append(result)

                except Exception as error:
                    results.append({
                        "segment_col": f"{first_segment}+{second_segment}",
                        "segment_value": f"{combo[0]}+{combo[1]}",
                        "segment_n": int(len(subset)),
                        "error": str(error)
                    })

    return results


def calculate_business_impact(
    statistical_result: Optional[Dict[str, Any]],
    config: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Calculate basic business impact when value and cost assumptions are available.

    For binary outcomes:
    - control expected value = control_rate * value per success
    - treatment expected value = treatment_rate * (value per success - intervention cost per success)
    - difference = treatment expected value - control expected value

    This is a simplified first-pass calculation for decision review, not a full finance model.
    """
    if statistical_result is None:
        return {
            "available": False,
            "reason": "No statistical result available."
        }

    value = config.get("business_value_per_success")
    cost = config.get("intervention_cost_per_success")

    if value is None:
        return {
            "available": False,
            "reason": "Business value per success was not provided."
        }

    method = statistical_result.get("method")

    if method == "two_proportion_test":
        control_rate = statistical_result["control_rate"]
        treatment_rate = statistical_result["treatment_rate"]

        cost = cost if cost is not None else 0

        control_expected_value = control_rate * value
        treatment_expected_value = treatment_rate * (value - cost)
        difference = treatment_expected_value - control_expected_value

        return {
            "available": True,
            "method": "binary_outcome_expected_value",
            "business_value_per_success": float(value),
            "intervention_cost_per_success": float(cost),
            "control_expected_value_per_unit": float(control_expected_value),
            "treatment_expected_value_per_unit": float(treatment_expected_value),
            "incremental_expected_value_per_unit": float(difference),
            "financially_positive": bool(difference > 0)
        }

    return {
        "available": False,
        "reason": "Business impact calculation is currently implemented for binary outcome tests only."
    }


def calculate_segment_business_impact(
    segment_results: List[Dict[str, Any]],
    config: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """
    Apply the business impact calculation to each segment result.
    """
    enriched_segments = []

    for result in segment_results:
        if "error" in result:
            enriched_segments.append(result)
            continue

        impact = calculate_business_impact(result, config)

        enriched = result.copy()
        enriched["business_impact"] = impact
        enriched_segments.append(enriched)

    return enriched_segments


def summarize_segment_results(segment_results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Summarize segment effects to identify strongest and weakest segments.
    """
    valid_results = [
        r for r in segment_results
        if "effect_pp" in r and "error" not in r
    ]

    if len(valid_results) == 0:
        return {
            "available": False,
            "reason": "No valid segment effects were calculated."
        }

    strongest = max(valid_results, key=lambda r: r.get("effect_pp", float("-inf")))
    weakest = min(valid_results, key=lambda r: r.get("effect_pp", float("inf")))

    financially_positive_segments = []
    for result in valid_results:
        impact = result.get("business_impact", {})
        if impact.get("available") and impact.get("financially_positive"):
            financially_positive_segments.append({
                "segment_col": result.get("segment_col"),
                "segment_value": result.get("segment_value"),
                "incremental_expected_value_per_unit": impact.get("incremental_expected_value_per_unit"),
                "effect_pp": result.get("effect_pp")
            })

    return {
        "available": True,
        "n_segments_analyzed": len(valid_results),
        "strongest_segment": {
            "segment_col": strongest.get("segment_col"),
            "segment_value": strongest.get("segment_value"),
            "effect_pp": strongest.get("effect_pp"),
            "p_value": strongest.get("p_value")
        },
        "weakest_segment": {
            "segment_col": weakest.get("segment_col"),
            "segment_value": weakest.get("segment_value"),
            "effect_pp": weakest.get("effect_pp"),
            "p_value": weakest.get("p_value")
        },
        "financially_positive_segments": financially_positive_segments
    }


def run_segment_and_business_review(
    df: pd.DataFrame,
    config: Dict[str, Any],
    validation_package: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Combine overall business impact, segment effects, and segment-level business impact.
    """
    statistical_result = validation_package.get("statistical_result")

    overall_business_impact = calculate_business_impact(statistical_result, config)

    segment_results = run_segment_effects(df, config)
    segment_results = calculate_segment_business_impact(segment_results, config)

    segment_summary = summarize_segment_results(segment_results)

    return {
        "overall_business_impact": overall_business_impact,
        "segment_results": segment_results,
        "segment_summary": segment_summary
    }


def print_business_review(review: Dict[str, Any]) -> None:
    """
    Display business and segment review results in readable form.
    """
    print("SEGMENT AND BUSINESS IMPACT REVIEW")
    print("=" * 70)

    print("\nOverall business impact:")
    print(json.dumps(review["overall_business_impact"], indent=2))

    print("\nSegment summary:")
    print(json.dumps(review["segment_summary"], indent=2))

    if len(review["segment_results"]) > 0:
        print("\nSegment results preview:")
        preview_cols = [
            "segment_col",
            "segment_value",
            "segment_n",
            "control_rate",
            "treatment_rate",
            "effect_pp",
            "p_value"
        ]

        segment_df = pd.DataFrame(review["segment_results"])
        available_cols = [col for col in preview_cols if col in segment_df.columns]
        display(segment_df[available_cols].head(20))
    else:
        print("\nNo segment results available.")


print("Segment and Business Impact Tools defined.")

Segment and Business Impact Tools defined.


## 3.5 Claim Type, Evidence Tier, and Exception Router

The Validity Guard reviews the dataset profile, method decision, statistical validation package, and business impact review to decide what kind of claim is being made and what level of evidence supports it.

ClaimCheck separates **claim type** from **evidence tier**.

A claim type describes the business intent of the statement, such as descriptive, diagnostic, predictive, causal, comparative, segment-targeting, prescriptive, anomaly-monitoring, forecasting, or ROI/business-case. An evidence tier describes how strongly the available data supports that claim.

This distinction matters because business teams often make recommendations that are stronger than the evidence supports. For example, a campaign may show statistically significant lift, but if broad rollout is financially negative, the evidence may support a targeted recommendation rather than a broad rollout.

In this prototype, the Validity Guard assigns one of five evidence tiers:

1. **Descriptive:** The data can describe what happened, but cannot explain why.
2. **Predictive:** The data supports prediction, risk scoring, or driver interpretation, but not causal language.
3. **Suggestive:** The data shows a relationship or directional pattern, but stronger causal or business validation is needed.
4. **Causal:** A valid treatment/control pathway supports a treatment-effect claim.
5. **Decision-Ready:** The evidence supports both the analytical claim and the recommended business action.

The Exception Router decides whether human review is required. Human review is triggered when the method is unsupported, configured data is missing, treatment/control balance is questionable, segment results conflict with the overall result, business impact is negative despite statistical lift, or the requested claim strength exceeds the evidence tier.

In [7]:
# ============================================================
# 3.5: Claim Type, Evidence Tier, and Exception Router
# ============================================================

def classify_claim_type(config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Classify the business intent of the claim.
    This is a rule-based prototype of what the Claim Framer agent would do with an LLM.
    """
    claim_text = str(config.get("claim", "")).lower()
    decision_context = str(config.get("decision_context", "")).lower()
    combined_text = f"{claim_text} {decision_context}"

    category_rules = {
        "prescriptive_recommendation": [
            "should", "recommend", "roll out", "rollout", "launch", "scale",
            "target", "pause", "continue", "invest", "prioritize"
        ],
        "causal_impact": [
            "caused", "drove", "increased", "reduced", "improved", "lifted",
            "impact", "effect", "because of", "due to"
        ],
        "predictive": [
            "predict", "prediction", "likely", "risk", "propensity", "score",
            "forecast", "probability", "churn risk"
        ],
        "diagnostic": [
            "why", "driver", "drivers", "explains", "reason", "root cause",
            "caused by", "attributable"
        ],
        "comparative": [
            "better than", "worse than", "outperformed", "underperformed",
            "higher than", "lower than", "variant", "group", "compared"
        ],
        "segment_targeting": [
            "segment", "target segment", "female", "male", "gamers",
            "non-gamers", "customers like", "cohort"
        ],
        "anomaly_monitoring": [
            "spike", "drop", "anomaly", "unusual", "alert", "sudden",
            "outlier", "unexpected"
        ],
        "roi_business_case": [
            "profit", "profitable", "revenue", "cost", "roi", "expected value",
            "margin", "payback", "financial"
        ],
        "forecasting": [
            "forecast", "next quarter", "next month", "future", "will grow",
            "projected"
        ],
        "descriptive": [
            "increased", "decreased", "was", "is", "observed", "showed"
        ]
    }

    detected_types = []

    for claim_type, keywords in category_rules.items():
        if any(keyword in combined_text for keyword in keywords):
            detected_types.append(claim_type)

    # Prescriptive claims often contain causal/comparative/ROI language too.
    # Keep all detected types, but choose a primary type for routing.
    priority_order = [
        "prescriptive_recommendation",
        "causal_impact",
        "roi_business_case",
        "segment_targeting",
        "predictive",
        "diagnostic",
        "comparative",
        "anomaly_monitoring",
        "forecasting",
        "descriptive"
    ]

    primary_type = "descriptive"
    for claim_type in priority_order:
        if claim_type in detected_types:
            primary_type = claim_type
            break

    return {
        "primary_claim_type": primary_type,
        "detected_claim_types": detected_types if detected_types else ["descriptive"]
    }


def assign_evidence_tier(
    profile: Dict[str, Any],
    method_decision: Dict[str, Any],
    validation_package: Dict[str, Any],
    business_review: Dict[str, Any],
    config: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Assign an evidence tier based on data quality, method support,
    statistical validation, balance checks, and business impact.
    """
    claim_type_package = classify_claim_type(config)
    method_id = method_decision.get("method_id")
    requested_strength = config.get("requested_claim_strength", "causal").lower()

    reasons = []
    risk_flags = []

    # Default tier
    evidence_tier = "Descriptive"

    # Data/config problems force descriptive tier.
    if profile.get("flags"):
        risk_flags.extend(profile.get("flags", []))

    if not profile.get("config_validation", {}).get("all_configured_columns_found", False):
        reasons.append("Configured columns are missing, so the claim cannot be fully validated.")
        return {
            "claim_type": claim_type_package,
            "evidence_tier": "Descriptive",
            "requested_claim_strength": requested_strength,
            "risk_flags": risk_flags,
            "reasons": reasons,
            "causal_language_allowed": False,
            "decision_language_allowed": False
        }

    if validation_package.get("ran_successfully") is False:
        reasons.append("Selected validation pathway did not run successfully.")
        return {
            "claim_type": claim_type_package,
            "evidence_tier": "Descriptive",
            "requested_claim_strength": requested_strength,
            "risk_flags": risk_flags,
            "reasons": reasons,
            "causal_language_allowed": False,
            "decision_language_allowed": False
        }

    # Predictive and descriptive routes.
    if method_id == "predictive_driver_review":
        evidence_tier = "Predictive"
        reasons.append("Predictive or driver-style evidence supports risk or association language, not causal language.")

    elif method_id == "descriptive_only":
        evidence_tier = "Descriptive"
        reasons.append("Descriptive route can summarize what happened but cannot support causal or decision-ready claims.")

    # Implemented treatment/control routes.
    elif method_id in ["two_proportion_test", "two_sample_t_test"]:
        statistical_result = validation_package.get("statistical_result")
        balance_result = validation_package.get("balance_result")

        if statistical_result is None:
            evidence_tier = "Descriptive"
            reasons.append("No statistical result was available.")
        else:
            is_significant = statistical_result.get("statistically_significant", False)

            if balance_result and balance_result.get("balanced") is False:
                evidence_tier = "Suggestive"
                risk_flags.append("Treatment/control groups are not balanced on configured covariates.")
                reasons.append("Balance checks failed, so causal confidence is downgraded.")

            elif balance_result and balance_result.get("balanced") is None:
                evidence_tier = "Suggestive"
                risk_flags.append("No valid covariates were available for balance checking.")
                reasons.append("Treatment/control structure exists, but balance could not be fully assessed.")

            elif is_significant:
                evidence_tier = "Causal"
                reasons.append("Implemented treatment/control validation ran successfully and the result is statistically significant.")

                if balance_result and balance_result.get("balanced") is True:
                    reasons.append("Configured covariates appear balanced between treatment and control groups.")

            else:
                evidence_tier = "Suggestive"
                risk_flags.append("Treatment effect is not statistically significant at the configured alpha.")
                reasons.append("The result does not provide strong statistical evidence of a treatment effect.")

    else:
        evidence_tier = "Suggestive"
        risk_flags.append(f"Selected method '{method_id}' is recognized but not implemented in this prototype.")
        reasons.append("Unsupported method is routed safely rather than executed.")

    # Business impact can upgrade from Causal to Decision-Ready, or block decision language.
    impact = business_review.get("overall_business_impact", {})

    if evidence_tier == "Causal" and impact.get("available"):
        if impact.get("financially_positive") is True:
            evidence_tier = "Decision-Ready"
            reasons.append("The result is statistically supported and overall business impact is financially positive.")
        elif impact.get("financially_positive") is False:
            risk_flags.append("Overall business impact is negative despite statistical lift.")
            reasons.append("Statistical improvement does not justify broad rollout because expected value is negative.")

    causal_language_allowed = evidence_tier in ["Causal", "Decision-Ready"]
    decision_language_allowed = evidence_tier == "Decision-Ready"

    return {
        "claim_type": claim_type_package,
        "evidence_tier": evidence_tier,
        "requested_claim_strength": requested_strength,
        "risk_flags": risk_flags,
        "reasons": reasons,
        "causal_language_allowed": causal_language_allowed,
        "decision_language_allowed": decision_language_allowed
    }


def route_exceptions(
    evidence_package: Dict[str, Any],
    method_decision: Dict[str, Any],
    validation_package: Dict[str, Any],
    business_review: Dict[str, Any],
    config: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Decide whether a human reviewer should inspect the case before final recommendation.
    """
    review_reasons = []

    evidence_tier = evidence_package.get("evidence_tier")
    requested_strength = evidence_package.get("requested_claim_strength", "causal")
    claim_type = evidence_package.get("claim_type", {}).get("primary_claim_type")

    if method_decision.get("human_review_required"):
        review_reasons.append("Method Selection Engine marked this case for human review.")

    if validation_package.get("ran_successfully") is False:
        review_reasons.append("Statistical validation did not run successfully.")

    if evidence_package.get("risk_flags"):
        review_reasons.extend(evidence_package.get("risk_flags", []))

    if requested_strength == "causal" and evidence_tier not in ["Causal", "Decision-Ready"]:
        review_reasons.append("Requested claim strength is causal, but evidence tier is not causal.")

    if claim_type == "prescriptive_recommendation" and evidence_tier != "Decision-Ready":
        review_reasons.append("Claim asks for a business recommendation, but evidence is not decision-ready.")

    impact = business_review.get("overall_business_impact", {})
    if impact.get("available") and impact.get("financially_positive") is False:
        review_reasons.append("Business impact is negative; rollout recommendation requires review.")

    segment_summary = business_review.get("segment_summary", {})
    if segment_summary.get("available"):
        positive_segments = segment_summary.get("financially_positive_segments", [])
        overall_positive = impact.get("financially_positive")

        if overall_positive is False and len(positive_segments) > 0:
            review_reasons.append("Some segments appear financially positive while overall rollout is negative.")

    # Remove duplicates while preserving order.
    review_reasons = list(dict.fromkeys(review_reasons))

    return {
        "human_review_required": len(review_reasons) > 0,
        "review_type": "exception_review" if len(review_reasons) > 0 else "auto_clear",
        "review_reasons": review_reasons
    }


def run_validity_guard(
    profile: Dict[str, Any],
    method_decision: Dict[str, Any],
    validation_package: Dict[str, Any],
    business_review: Dict[str, Any],
    config: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Combined Validity Guard output: claim type + evidence tier + exception routing.
    """
    evidence_package = assign_evidence_tier(
        profile=profile,
        method_decision=method_decision,
        validation_package=validation_package,
        business_review=business_review,
        config=config
    )

    exception_package = route_exceptions(
        evidence_package=evidence_package,
        method_decision=method_decision,
        validation_package=validation_package,
        business_review=business_review,
        config=config
    )

    return {
        "evidence_package": evidence_package,
        "exception_package": exception_package
    }


def print_validity_guard_output(validity_output: Dict[str, Any]) -> None:
    """
    Display Validity Guard output in a readable format.
    """
    print("VALIDITY GUARD OUTPUT")
    print("=" * 70)

    print("\nClaim type and evidence tier:")
    print(json.dumps(validity_output["evidence_package"], indent=2))

    print("\nException routing:")
    print(json.dumps(validity_output["exception_package"], indent=2))


print("Claim Type, Evidence Tier, and Exception Router defined.")

Claim Type, Evidence Tier, and Exception Router defined.


## 3.6 Tier-Locked Language Guardrail

The Language Guardrail prevents the final recommendation from using stronger language than the evidence supports.

This is necessary because business claims often become overstated when they move from analysis into executive communication. A dashboard pattern may become a causal claim, or a statistically significant lift may become a rollout recommendation without checking financial impact.

ClaimCheck applies tier-locked language rules:

- **Descriptive evidence** can describe what happened but cannot explain why it happened.
- **Predictive evidence** can discuss likelihood, risk, or association but cannot claim causality.
- **Suggestive evidence** can describe directional patterns but must avoid strong causal or decision language.
- **Causal evidence** can support treatment-effect language but may still require business-impact review.
- **Decision-Ready evidence** can support business recommendation language when both statistical and commercial checks pass.

This guardrail supports enterprise-safe analytics communication by rewriting or flagging unsupported wording before a claim reaches leadership.

In [8]:
# ============================================================
# ClaimCheck 3.6: Tier-Locked Language Guardrail
# ============================================================

BANNED_LANGUAGE_RULES = {
    "Descriptive": {
        "blocked_terms": [
            "caused", "drove", "proved", "resulted in", "led to",
            "because of", "due to", "impact", "effect", "recommend",
            "should roll out", "should launch", "decision-ready"
        ],
        "allowed_language": "Describe observed patterns only. Do not explain cause or recommend action."
    },
    "Predictive": {
        "blocked_terms": [
            "caused", "drove", "proved", "resulted in", "led to",
            "because of", "due to", "treatment effect", "causal impact",
            "should roll out", "decision-ready"
        ],
        "allowed_language": "Use prediction, risk, association, or likelihood language. Do not claim causality."
    },
    "Suggestive": {
        "blocked_terms": [
            "caused", "proved", "definitively", "guarantees",
            "decision-ready", "should roll out to all", "must launch"
        ],
        "allowed_language": "Use directional, associational, or further-testing language."
    },
    "Causal": {
        "blocked_terms": [
            "proved", "guarantees", "risk-free", "must launch",
            "should roll out to all"
        ],
        "allowed_language": "Causal treatment-effect language is allowed, but recommendation language still depends on business impact."
    },
    "Decision-Ready": {
        "blocked_terms": [
            "proved", "guarantees", "risk-free"
        ],
        "allowed_language": "Recommendation language is allowed, but uncertainty and assumptions should still be stated."
    }
}


LANGUAGE_REWRITE_MAP = {
    "caused": "is associated with",
    "drove": "is associated with",
    "proved": "provides evidence that",
    "resulted in": "is associated with",
    "led to": "is associated with",
    "because of": "alongside",
    "due to": "alongside",
    "guarantees": "suggests",
    "risk-free": "lower-risk",
    "must launch": "should be reviewed for launch",
    "should roll out to all": "should be reviewed before broad rollout"
}


def scan_for_blocked_language(text: str, evidence_tier: str) -> Dict[str, Any]:
    """
    Scan a text string for language that is not allowed under the assigned evidence tier.
    """
    text_lower = str(text).lower()

    rules = BANNED_LANGUAGE_RULES.get(evidence_tier, BANNED_LANGUAGE_RULES["Descriptive"])
    blocked_terms = rules["blocked_terms"]

    found_terms = [
        term for term in blocked_terms
        if term in text_lower
    ]

    return {
        "evidence_tier": evidence_tier,
        "blocked_terms_found": found_terms,
        "language_violation": len(found_terms) > 0,
        "allowed_language": rules["allowed_language"]
    }


def rewrite_claim_language(text: str, evidence_tier: str) -> Dict[str, Any]:
    """
    Rewrite unsupported language using softer evidence-appropriate alternatives.
    This is a deterministic prototype. In CrewAI, the Brief Builder agent can use these rules.
    """
    scan = scan_for_blocked_language(text, evidence_tier)
    revised_text = str(text)

    for blocked_term in scan["blocked_terms_found"]:
        replacement = LANGUAGE_REWRITE_MAP.get(blocked_term, "suggests")
        revised_text = revised_text.replace(blocked_term, replacement)
        revised_text = revised_text.replace(blocked_term.title(), replacement.title())
        revised_text = revised_text.replace(blocked_term.upper(), replacement.upper())

    return {
        "original_text": text,
        "revised_text": revised_text,
        "scan": scan
    }


def generate_allowed_language_guidance(validity_output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Generate language guidance based on evidence tier and claim type.
    """
    evidence_package = validity_output["evidence_package"]
    evidence_tier = evidence_package["evidence_tier"]
    claim_type = evidence_package["claim_type"]["primary_claim_type"]

    rules = BANNED_LANGUAGE_RULES.get(evidence_tier, BANNED_LANGUAGE_RULES["Descriptive"])

    guidance = {
        "claim_type": claim_type,
        "evidence_tier": evidence_tier,
        "causal_language_allowed": evidence_package.get("causal_language_allowed", False),
        "decision_language_allowed": evidence_package.get("decision_language_allowed", False),
        "allowed_language": rules["allowed_language"],
        "blocked_terms": rules["blocked_terms"]
    }

    if claim_type == "prescriptive_recommendation" and evidence_tier != "Decision-Ready":
        guidance["special_warning"] = (
            "The claim asks for a business recommendation, but the evidence is not decision-ready. "
            "Recommendation language should be softened or routed to human review."
        )

    elif claim_type == "predictive" and evidence_tier == "Predictive":
        guidance["special_warning"] = (
            "Predictive evidence can support risk or likelihood language, but not causal explanation."
        )

    elif claim_type == "causal_impact" and evidence_tier not in ["Causal", "Decision-Ready"]:
        guidance["special_warning"] = (
            "The claim uses impact-oriented language, but the evidence tier does not support a causal claim."
        )

    else:
        guidance["special_warning"] = None

    return guidance


def print_language_guardrail(guidance: Dict[str, Any]) -> None:
    """
    Display language guardrail guidance in a readable format.
    """
    print("TIER-LOCKED LANGUAGE GUARDRAIL")
    print("=" * 70)
    print(json.dumps(guidance, indent=2))


print("Tier-Locked Language Guardrail defined.")

Tier-Locked Language Guardrail defined.


## 3.7 Executive Brief Generator

The Brief Builder converts ClaimCheck's analytical outputs into an executive-facing recommendation.

This section is still deterministic in the notebook so that the output is reproducible. In the full CrewAI version, the Brief Builder agent would use the same structured outputs and language guardrail to write a more polished executive memo.

The brief includes:

- the original claim;
- selected claim type;
- evidence tier;
- method selected;
- statistical result summary;
- business impact summary;
- segment insight;
- human review status;
- final recommendation.

The key design principle is that the brief must use evidence-appropriate language. A descriptive or predictive case should not be written as a causal conclusion. A causal result should not automatically become a rollout recommendation unless business impact also supports it.

In [9]:
# ============================================================
# ClaimCheck 3.7: Executive Brief Generator
# ============================================================

def format_percentage(value: Optional[float], decimals: int = 2) -> str:
    """
    Format a decimal as a percentage string.
    """
    if value is None:
        return "not available"
    return f"{value:.{decimals}f}%"


def format_currency(value: Optional[float], decimals: int = 2) -> str:
    """
    Format a numeric value as a currency-style string.
    """
    if value is None:
        return "not available"
    return f"${value:.{decimals}f}"


def summarize_statistical_result(validation_package: Dict[str, Any]) -> str:
    """
    Create a concise statistical summary from the validation package.
    """
    result = validation_package.get("statistical_result")

    if result is None:
        return "No inferential statistical test was run for this case."

    method = result.get("method")

    if method == "two_proportion_test":
        return (
            f"The treatment group had a {result['treatment_rate'] * 100:.2f}% outcome rate "
            f"versus {result['control_rate'] * 100:.2f}% in the control group. "
            f"The estimated lift was {result['effect_pp']:.2f} percentage points "
            f"with p-value {result['p_value']:.4f}."
        )

    if method == "two_sample_t_test":
        return (
            f"The treatment group mean was {result['treatment_mean']:.2f} "
            f"versus {result['control_mean']:.2f} in the control group. "
            f"The estimated difference was {result['effect']:.2f} "
            f"with p-value {result['p_value']:.4f}."
        )

    return "A statistical result was generated, but no formatted summary is available for this method."


def summarize_business_impact(business_review: Dict[str, Any]) -> str:
    """
    Create a concise business impact summary.
    """
    impact = business_review.get("overall_business_impact", {})

    if not impact.get("available"):
        return f"Business impact was not calculated: {impact.get('reason', 'required assumptions were unavailable')}"

    incremental_value = impact.get("incremental_expected_value_per_unit")
    financially_positive = impact.get("financially_positive")

    direction = "positive" if financially_positive else "negative"

    return (
        f"The estimated incremental expected value per unit is "
        f"{format_currency(incremental_value)}. "
        f"The overall business impact is {direction} under the provided value and cost assumptions."
    )


def summarize_segment_insight(business_review: Dict[str, Any]) -> str:
    """
    Create a concise segment insight summary.
    """
    segment_summary = business_review.get("segment_summary", {})

    if not segment_summary.get("available"):
        return "No valid segment-level treatment effects were calculated."

    strongest = segment_summary.get("strongest_segment", {})
    weakest = segment_summary.get("weakest_segment", {})
    positive_segments = segment_summary.get("financially_positive_segments", [])

    text = (
        f"The strongest observed segment was {strongest.get('segment_value')} "
        f"within {strongest.get('segment_col')}, with estimated lift of "
        f"{strongest.get('effect_pp'):.2f} percentage points. "
        f"The weakest observed segment was {weakest.get('segment_value')} "
        f"within {weakest.get('segment_col')}, with estimated lift of "
        f"{weakest.get('effect_pp'):.2f} percentage points."
    )

    if len(positive_segments) > 0:
        segment_names = [
            f"{item['segment_col']}={item['segment_value']}"
            for item in positive_segments[:5]
        ]
        text += (
            " Financially positive segments were identified: "
            + ", ".join(segment_names)
            + "."
        )
    else:
        text += " No financially positive segment was identified under the current assumptions."

    return text


def determine_recommendation(
    validity_output: Dict[str, Any],
    business_review: Dict[str, Any]
) -> str:
    """
    Determine a recommendation status based on evidence tier, business impact, and human review.
    """
    evidence_package = validity_output["evidence_package"]
    exception_package = validity_output["exception_package"]

    evidence_tier = evidence_package.get("evidence_tier")
    human_review_required = exception_package.get("human_review_required")

    impact = business_review.get("overall_business_impact", {})
    segment_summary = business_review.get("segment_summary", {})

    if evidence_tier == "Decision-Ready" and not human_review_required:
        return "Approve recommendation based on current evidence."

    if evidence_tier in ["Causal", "Decision-Ready"] and impact.get("financially_positive") is False:
        positive_segments = segment_summary.get("financially_positive_segments", [])

        if len(positive_segments) > 0:
            return (
                "Do not approve broad rollout. Consider targeted rollout only for financially positive segments, "
                "subject to human review."
            )

        return "Do not approve rollout under the current business-impact assumptions."

    if evidence_tier == "Causal":
        return "Causal evidence is present, but business recommendation requires further review."

    if evidence_tier == "Predictive":
        return "Use as predictive or risk-ranking evidence only. Do not use as causal justification."

    if evidence_tier == "Suggestive":
        return "Treat as suggestive evidence. Run a stronger test or request human review before action."

    return "Use for descriptive reporting only. Do not use as causal or decision-ready evidence."


def generate_executive_brief(
    config: Dict[str, Any],
    profile: Dict[str, Any],
    method_decision: Dict[str, Any],
    validation_package: Dict[str, Any],
    business_review: Dict[str, Any],
    validity_output: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Generate an executive-facing brief using tier-locked language.
    """
    evidence_package = validity_output["evidence_package"]
    exception_package = validity_output["exception_package"]

    language_guidance = generate_allowed_language_guidance(validity_output)

    recommendation = determine_recommendation(validity_output, business_review)

    brief_text = f"""
ClaimCheck Executive Brief

Original claim:
{config.get("claim")}

Claim type:
{evidence_package["claim_type"]["primary_claim_type"]}

Evidence tier:
{evidence_package["evidence_tier"]}

Selected method:
{method_decision.get("method_id")} ({method_decision.get("method_status")})

Statistical summary:
{summarize_statistical_result(validation_package)}

Business impact summary:
{summarize_business_impact(business_review)}

Segment insight:
{summarize_segment_insight(business_review)}

Human review status:
{"Human review required" if exception_package["human_review_required"] else "Auto-clear"}

Review reasons:
{"; ".join(exception_package["review_reasons"]) if exception_package["review_reasons"] else "No exception review reasons triggered."}

Recommendation:
{recommendation}

Language guidance:
{language_guidance["allowed_language"]}
""".strip()

    rewritten = rewrite_claim_language(
        text=brief_text,
        evidence_tier=evidence_package["evidence_tier"]
    )

    return {
        "brief_text": rewritten["revised_text"],
        "original_brief_text": brief_text,
        "language_scan": rewritten["scan"],
        "language_guidance": language_guidance,
        "recommendation": recommendation
    }


def print_executive_brief(brief: Dict[str, Any]) -> None:
    """
    Display the final executive brief.
    """
    print("EXECUTIVE BRIEF")
    print("=" * 70)
    print(brief["brief_text"])

    print("\nLANGUAGE SCAN")
    print("=" * 70)
    print(json.dumps(brief["language_scan"], indent=2))


print("Executive Brief Generator defined.")

Executive Brief Generator defined.


## 3.8 Audit Logger and End-to-End ClaimCheck Runner

This section combines the individual deterministic tools into one end-to-end ClaimCheck workflow.

The runner executes the full review sequence:

1. Profile the dataset.
2. Select the analytical method.
3. Run statistical validation or safe routing.
4. Run segment and business-impact review.
5. Assign claim type and evidence tier.
6. Apply exception routing.
7. Generate an executive brief.
8. Create an audit log.

The audit log is central to enterprise readiness. It records the claim, dataset hash, method decision, evidence tier, human review status, recommendation, and timestamp. This allows an organization to trace how ClaimCheck reached a recommendation and which version of the dataset was used.


In [10]:
# ============================================================
# ClaimCheck 3.8: Audit Logger and End-to-End Runner
# ============================================================

def create_audit_log(
    config: Dict[str, Any],
    profile: Dict[str, Any],
    method_decision: Dict[str, Any],
    validation_package: Dict[str, Any],
    business_review: Dict[str, Any],
    validity_output: Dict[str, Any],
    brief: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Create an audit log for the ClaimCheck review.
    """
    run_id = f"claimcheck_{dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"

    evidence_package = validity_output["evidence_package"]
    exception_package = validity_output["exception_package"]

    audit_log = {
        "run_id": run_id,
        "timestamp_utc": dt.datetime.utcnow().isoformat(),
        "dataset_name": config.get("dataset_name"),
        "dataset_hash": profile.get("dataset_hash"),
        "original_claim": config.get("claim"),
        "decision_context": config.get("decision_context"),
        "primary_claim_type": evidence_package["claim_type"]["primary_claim_type"],
        "detected_claim_types": evidence_package["claim_type"]["detected_claim_types"],
        "selected_method": method_decision.get("method_id"),
        "method_status": method_decision.get("method_status"),
        "method_reason": method_decision.get("reason"),
        "statistical_validation_ran": validation_package.get("ran_successfully"),
        "evidence_tier": evidence_package.get("evidence_tier"),
        "causal_language_allowed": evidence_package.get("causal_language_allowed"),
        "decision_language_allowed": evidence_package.get("decision_language_allowed"),
        "human_review_required": exception_package.get("human_review_required"),
        "review_type": exception_package.get("review_type"),
        "review_reasons": exception_package.get("review_reasons"),
        "recommendation": brief.get("recommendation"),
        "language_violation": brief.get("language_scan", {}).get("language_violation"),
        "blocked_terms_found": brief.get("language_scan", {}).get("blocked_terms_found")
    }

    return audit_log


def run_claimcheck(
    df: pd.DataFrame,
    config: Dict[str, Any],
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Run the complete ClaimCheck workflow from raw dataset + claim config to final brief and audit log.
    """
    profile = profile_dataset(df, config)
    method_decision = select_method(profile, config)
    validation_package = run_statistical_validation(df, config, method_decision)
    business_review = run_segment_and_business_review(df, config, validation_package)

    validity_output = run_validity_guard(
        profile=profile,
        method_decision=method_decision,
        validation_package=validation_package,
        business_review=business_review,
        config=config
    )

    brief = generate_executive_brief(
        config=config,
        profile=profile,
        method_decision=method_decision,
        validation_package=validation_package,
        business_review=business_review,
        validity_output=validity_output
    )

    audit_log = create_audit_log(
        config=config,
        profile=profile,
        method_decision=method_decision,
        validation_package=validation_package,
        business_review=business_review,
        validity_output=validity_output,
        brief=brief
    )

    output = {
        "profile": profile,
        "method_decision": method_decision,
        "validation_package": validation_package,
        "business_review": business_review,
        "validity_output": validity_output,
        "executive_brief": brief,
        "audit_log": audit_log
    }

    if verbose:
        print_dataset_profile(profile)
        print("\n")
        print_method_decision(method_decision)
        print("\n")
        print_validation_package(validation_package)
        print("\n")
        print_business_review(business_review)
        print("\n")
        print_validity_guard_output(validity_output)
        print("\n")
        print_executive_brief(brief)
        print("\n")
        print("AUDIT LOG")
        print("=" * 70)
        print(json.dumps(audit_log, indent=2))

    return output


def save_claimcheck_outputs(
    output: Dict[str, Any],
    output_prefix: str = "claimcheck_output"
) -> Dict[str, str]:
    """
    Save ClaimCheck outputs as JSON and text files for GitHub/demo evidence.
    """
    json_path = f"{output_prefix}.json"
    brief_path = f"{output_prefix}_executive_brief.txt"

    with open(json_path, "w") as f:
        json.dump(output, f, indent=2, default=str)

    with open(brief_path, "w") as f:
        f.write(output["executive_brief"]["brief_text"])

    return {
        "json_path": json_path,
        "brief_path": brief_path
    }


print("Audit Logger and End-to-End ClaimCheck Runner defined.")

Audit Logger and End-to-End ClaimCheck Runner defined.


## 3.9 One-Click Evidence Report and Executive Deck Generator

This section turns ClaimCheck's deterministic review output into editable business artifacts.

A business analyst or product manager does not only need a JSON object or console output. They need evidence they can share: a memo, a deck, charts, recommendations, limitations, and an audit trail. ClaimCheck therefore creates a one-click artifact bundle after the validated workflow has completed.

The generator produces:

1. **Word evidence report** for analyst review, methodology, findings, limitations, and audit trail.
2. **PowerPoint executive deck** for stakeholder presentation.
3. **Charts** showing treatment/control comparison, segment lift, and business impact where available.
4. **JSON output and executive brief text** for reproducibility and downstream integration.
5. **ZIP bundle** for demo and GitHub packaging.

The important design principle is that these artifacts are not generated directly from free-form prompting. They are generated from ClaimCheck's structured outputs: dataset profile, method selection, statistical validation, business impact, evidence tier, human review routing, language guardrails, and audit log.

In [11]:
# ============================================================
# ClaimCheck 3.9: One-Click Evidence Report and Executive Deck Generator
# ============================================================

import os
import zipfile
import matplotlib.pyplot as plt

try:
    from docx import Document
    from docx.shared import Inches
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
    from docx import Document
    from docx.shared import Inches

try:
    from pptx import Presentation
    from pptx.util import Inches as PPTInches, Pt
    from pptx.enum.text import PP_ALIGN
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx"])
    from pptx import Presentation
    from pptx.util import Inches as PPTInches, Pt
    from pptx.enum.text import PP_ALIGN


# ------------------------------------------------------------
# Helper formatting functions
# ------------------------------------------------------------

def clean_text(value: Any) -> str:
    """
    Convert values into readable text for reports and slides.
    """
    if value is None:
        return "Not available"
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def pct(value: Optional[float], decimals: int = 2) -> str:
    """
    Format decimal values as percentages.
    """
    if value is None:
        return "Not available"
    return f"{value * 100:.{decimals}f}%"


def money(value: Optional[float], decimals: int = 2) -> str:
    """
    Format numeric values as currency.
    """
    if value is None:
        return "Not available"
    return f"${value:.{decimals}f}"


# ------------------------------------------------------------
# Chart generation
# ------------------------------------------------------------

def create_claimcheck_charts(
    output: Dict[str, Any],
    output_dir: str = "claimcheck_charts"
) -> Dict[str, str]:
    """
    Generate reusable charts from ClaimCheck output.
    These charts can be inserted into Word reports, PowerPoint decks, or Streamlit.
    """
    os.makedirs(output_dir, exist_ok=True)
    chart_paths = {}

    validation_package = output.get("validation_package", {})
    business_review = output.get("business_review", {})

    stat_result = validation_package.get("statistical_result")
    segment_results = business_review.get("segment_results", [])

    # Chart 1: Treatment vs control
    if stat_result is not None:
        method = stat_result.get("method")

        if method == "two_proportion_test":
            labels = ["Control", "Treatment"]
            values = [
                stat_result.get("control_rate", 0) * 100,
                stat_result.get("treatment_rate", 0) * 100
            ]
            ylabel = "Outcome rate (%)"
            title = "Treatment vs Control Outcome Rate"

        elif method == "two_sample_t_test":
            labels = ["Control", "Treatment"]
            values = [
                stat_result.get("control_mean", 0),
                stat_result.get("treatment_mean", 0)
            ]
            ylabel = "Mean outcome"
            title = "Treatment vs Control Mean Outcome"

        else:
            labels, values, ylabel, title = [], [], "", ""

        if labels:
            plt.figure(figsize=(6.5, 4.2))
            plt.bar(labels, values)
            plt.ylabel(ylabel)
            plt.title(title)
            plt.tight_layout()

            path = os.path.join(output_dir, "treatment_vs_control.png")
            plt.savefig(path, dpi=160)
            plt.close()
            chart_paths["treatment_vs_control"] = path

    # Chart 2: Segment-level lift
    valid_segments = [
        row for row in segment_results
        if "effect_pp" in row and "error" not in row
    ]

    if len(valid_segments) > 0:
        segment_df = pd.DataFrame(valid_segments).copy()
        segment_df["segment_label"] = (
            segment_df["segment_col"].astype(str)
            + "="
            + segment_df["segment_value"].astype(str)
        )

        segment_df = segment_df.sort_values("effect_pp", ascending=False).head(10)

        plt.figure(figsize=(8.5, 5))
        plt.barh(segment_df["segment_label"], segment_df["effect_pp"])
        plt.xlabel("Estimated lift / difference")
        plt.title("Segment-Level Treatment Effect")
        plt.gca().invert_yaxis()
        plt.tight_layout()

        path = os.path.join(output_dir, "segment_lift.png")
        plt.savefig(path, dpi=160)
        plt.close()
        chart_paths["segment_lift"] = path

    # Chart 3: Segment-level business impact
    business_rows = []

    for row in valid_segments:
        impact = row.get("business_impact", {})
        if impact.get("available"):
            business_rows.append({
                "segment_label": f"{row.get('segment_col')}={row.get('segment_value')}",
                "incremental_expected_value_per_unit": impact.get("incremental_expected_value_per_unit")
            })

    if len(business_rows) > 0:
        impact_df = pd.DataFrame(business_rows)
        impact_df = impact_df.sort_values(
            "incremental_expected_value_per_unit",
            ascending=False
        ).head(10)

        plt.figure(figsize=(8.5, 5))
        plt.barh(
            impact_df["segment_label"],
            impact_df["incremental_expected_value_per_unit"]
        )
        plt.xlabel("Incremental expected value per unit")
        plt.title("Business Impact by Segment")
        plt.gca().invert_yaxis()
        plt.tight_layout()

        path = os.path.join(output_dir, "segment_business_impact.png")
        plt.savefig(path, dpi=160)
        plt.close()
        chart_paths["segment_business_impact"] = path

    return chart_paths


# ------------------------------------------------------------
# Word report generation
# ------------------------------------------------------------

def add_word_table(document: Document, title: str, data: Dict[str, Any]) -> None:
    """
    Add a clean key-value table to the Word report.
    """
    document.add_heading(title, level=2)

    table = document.add_table(rows=1, cols=2)
    table.style = "Table Grid"

    header = table.rows[0].cells
    header[0].text = "Field"
    header[1].text = "Value"

    for key, value in data.items():
        row = table.add_row().cells
        row[0].text = clean_text(key)
        row[1].text = clean_text(value)


def generate_claimcheck_word_report(
    output: Dict[str, Any],
    report_path: str = "ClaimCheck_Evidence_Report.docx",
    report_length: str = "full_packet"
) -> str:
    """
    Generate an editable Word evidence report.

    report_length options:
    - decision_memo: short decision memo
    - evidence_memo: medium analyst memo
    - full_packet: detailed evidence packet with audit trail
    """
    chart_paths = create_claimcheck_charts(output)

    profile = output["profile"]
    method_decision = output["method_decision"]
    validation_package = output["validation_package"]
    business_review = output["business_review"]
    validity_output = output["validity_output"]
    executive_brief = output["executive_brief"]
    audit_log = output["audit_log"]

    evidence_package = validity_output["evidence_package"]
    exception_package = validity_output["exception_package"]

    document = Document()

    document.add_heading("ClaimCheck Evidence Report", level=1)

    document.add_paragraph(
        "ClaimCheck evaluates whether a business claim is supported by the available data, "
        "selected method, statistical evidence, business impact, evidence tier, language guardrails, "
        "human review routing, and audit trail."
    )

    document.add_heading("1. Executive Verdict", level=2)
    document.add_paragraph(executive_brief.get("recommendation", "No recommendation available."))

    document.add_heading("2. Submitted Claim", level=2)
    document.add_paragraph(clean_text(audit_log.get("original_claim")))

    add_word_table(
        document,
        "3. Evidence Governance Summary",
        {
            "Primary claim type": evidence_package["claim_type"]["primary_claim_type"],
            "Detected claim types": ", ".join(evidence_package["claim_type"]["detected_claim_types"]),
            "Evidence tier": evidence_package.get("evidence_tier"),
            "Selected method": audit_log.get("selected_method"),
            "Human review required": exception_package.get("human_review_required"),
            "Causal language allowed": evidence_package.get("causal_language_allowed"),
            "Decision language allowed": evidence_package.get("decision_language_allowed")
        }
    )

    if report_length in ["evidence_memo", "full_packet"]:
        add_word_table(
            document,
            "4. Dataset Readiness",
            {
                "Dataset name": audit_log.get("dataset_name"),
                "Dataset hash": audit_log.get("dataset_hash"),
                "Rows": profile.get("n_rows"),
                "Columns": profile.get("n_columns"),
                "Outcome type": profile.get("outcome_type"),
                "Two-group treatment structure": profile.get("has_two_group_treatment"),
                "Profile flags": "; ".join(profile.get("flags", [])) if profile.get("flags") else "None"
            }
        )

        add_word_table(
            document,
            "5. Method Selection",
            {
                "Selected method": method_decision.get("method_id"),
                "Method status": method_decision.get("method_status"),
                "Auto-runnable": method_decision.get("auto_runnable"),
                "Reason": method_decision.get("reason")
            }
        )

        document.add_heading("6. Statistical Evidence", level=2)
        document.add_paragraph(summarize_statistical_result(validation_package))

        if "treatment_vs_control" in chart_paths:
            document.add_picture(chart_paths["treatment_vs_control"], width=Inches(5.8))

        document.add_heading("7. Business Impact", level=2)
        document.add_paragraph(summarize_business_impact(business_review))

        document.add_heading("8. Segment Analysis", level=2)
        document.add_paragraph(summarize_segment_insight(business_review))

        if "segment_lift" in chart_paths:
            document.add_picture(chart_paths["segment_lift"], width=Inches(6.2))

        if "segment_business_impact" in chart_paths:
            document.add_picture(chart_paths["segment_business_impact"], width=Inches(6.2))

    if report_length == "full_packet":
        document.add_heading("9. Review Reasons and Limitations", level=2)
        review_reasons = exception_package.get("review_reasons", [])

        if review_reasons:
            for reason in review_reasons:
                document.add_paragraph(reason, style="List Bullet")
        else:
            document.add_paragraph("No exception review reasons were triggered.")

        document.add_heading("10. Executive Brief", level=2)
        document.add_paragraph(executive_brief.get("brief_text", ""))

        add_word_table(
            document,
            "11. Audit Trail",
            {
                "Run ID": audit_log.get("run_id"),
                "Timestamp UTC": audit_log.get("timestamp_utc"),
                "Dataset hash": audit_log.get("dataset_hash"),
                "Selected method": audit_log.get("selected_method"),
                "Evidence tier": audit_log.get("evidence_tier"),
                "Human review required": audit_log.get("human_review_required"),
                "Language violation": audit_log.get("language_violation"),
                "Blocked terms found": ", ".join(audit_log.get("blocked_terms_found", [])) if audit_log.get("blocked_terms_found") else "None"
            }
        )

    document.save(report_path)
    return report_path


# ------------------------------------------------------------
# PowerPoint deck generation
# ------------------------------------------------------------

def ppt_add_title(slide, title: str, subtitle: Optional[str] = None) -> None:
    """
    Add a title and optional subtitle to a PowerPoint slide.
    """
    title_box = slide.shapes.add_textbox(
        PPTInches(0.5), PPTInches(0.25), PPTInches(12.3), PPTInches(0.55)
    )
    frame = title_box.text_frame
    frame.clear()

    p = frame.paragraphs[0]
    p.text = title
    p.font.size = Pt(26)
    p.font.bold = True

    if subtitle:
        subtitle_box = slide.shapes.add_textbox(
            PPTInches(0.5), PPTInches(0.85), PPTInches(12.3), PPTInches(0.35)
        )
        subframe = subtitle_box.text_frame
        subframe.clear()

        sp = subframe.paragraphs[0]
        sp.text = subtitle
        sp.font.size = Pt(12)


def ppt_add_bullets(
    slide,
    bullets: List[str],
    left: float = 0.75,
    top: float = 1.4,
    width: float = 11.7,
    height: float = 4.8,
    font_size: int = 16
) -> None:
    """
    Add bullet points to a slide.
    """
    box = slide.shapes.add_textbox(
        PPTInches(left), PPTInches(top), PPTInches(width), PPTInches(height)
    )
    frame = box.text_frame
    frame.clear()
    frame.word_wrap = True

    for i, bullet in enumerate(bullets):
        p = frame.paragraphs[0] if i == 0 else frame.add_paragraph()
        p.text = clean_text(bullet)
        p.font.size = Pt(font_size)
        p.level = 0


def ppt_add_metric_cards(slide, metrics: Dict[str, Any], left: float = 0.6, top: float = 1.35) -> None:
    """
    Add simple metric cards to a PowerPoint slide.
    """
    card_width = 3.05
    card_height = 0.9
    gap = 0.22

    for idx, (label, value) in enumerate(metrics.items()):
        x = left + (idx % 4) * (card_width + gap)
        y = top + (idx // 4) * (card_height + gap)

        shape = slide.shapes.add_shape(
            1,
            PPTInches(x),
            PPTInches(y),
            PPTInches(card_width),
            PPTInches(card_height)
        )

        frame = shape.text_frame
        frame.clear()

        p1 = frame.paragraphs[0]
        p1.text = clean_text(value)
        p1.font.size = Pt(18)
        p1.font.bold = True
        p1.alignment = PP_ALIGN.CENTER

        p2 = frame.add_paragraph()
        p2.text = clean_text(label)
        p2.font.size = Pt(10)
        p2.alignment = PP_ALIGN.CENTER


def ppt_add_image(slide, image_path: Optional[str], left: float, top: float, width: float) -> None:
    """
    Add image to slide if image exists.
    """
    if image_path and os.path.exists(image_path):
        slide.shapes.add_picture(
            image_path,
            PPTInches(left),
            PPTInches(top),
            width=PPTInches(width)
        )


def generate_claimcheck_powerpoint_deck(
    output: Dict[str, Any],
    deck_path: str = "ClaimCheck_Executive_Deck.pptx",
    deck_style: str = "consulting_8_slide"
) -> str:
    """
    Generate an editable PowerPoint executive deck.

    deck_style options:
    - executive_5_slide
    - consulting_8_slide
    """
    chart_paths = create_claimcheck_charts(output)

    profile = output["profile"]
    method_decision = output["method_decision"]
    validation_package = output["validation_package"]
    business_review = output["business_review"]
    validity_output = output["validity_output"]
    executive_brief = output["executive_brief"]
    audit_log = output["audit_log"]

    evidence_package = validity_output["evidence_package"]
    exception_package = validity_output["exception_package"]
    stat_result = validation_package.get("statistical_result")
    impact = business_review.get("overall_business_impact", {})
    segment_summary = business_review.get("segment_summary", {})

    prs = Presentation()
    prs.slide_width = PPTInches(13.333)
    prs.slide_height = PPTInches(7.5)

    blank = prs.slide_layouts[6]

    # Slide 1: Verdict
    slide = prs.slides.add_slide(blank)
    ppt_add_title(slide, "ClaimCheck Executive Verdict", "Evidence-governed review of the submitted business claim")

    ppt_add_metric_cards(
        slide,
        {
            "Evidence tier": evidence_package.get("evidence_tier"),
            "Claim type": evidence_package["claim_type"]["primary_claim_type"],
            "Selected method": method_decision.get("method_id"),
            "Human review": "Required" if exception_package.get("human_review_required") else "Not required"
        }
    )

    ppt_add_bullets(
        slide,
        [
            f"Recommendation: {executive_brief.get('recommendation')}",
            f"Causal language allowed: {evidence_package.get('causal_language_allowed')}",
            f"Decision language allowed: {evidence_package.get('decision_language_allowed')}"
        ],
        top=3.0,
        font_size=17
    )

    # Slide 2: Claim and business question
    slide = prs.slides.add_slide(blank)
    ppt_add_title(slide, "Business Question and Submitted Claim", "What the analyst or product manager wants to validate")

    ppt_add_bullets(
        slide,
        [
            f"Submitted claim: {audit_log.get('original_claim')}",
            f"Decision context: {audit_log.get('decision_context')}",
            f"Detected claim types: {', '.join(evidence_package['claim_type']['detected_claim_types'])}",
            "ClaimCheck evaluates whether the claim is analytically supported, commercially justified, and safe for executive language."
        ],
        font_size=17
    )

    # Slide 3: Dataset readiness
    slide = prs.slides.add_slide(blank)
    ppt_add_title(slide, "Dataset Readiness", "Can the available data support the requested analysis?")

    ppt_add_metric_cards(
        slide,
        {
            "Rows": profile.get("n_rows"),
            "Columns": profile.get("n_columns"),
            "Outcome type": profile.get("outcome_type"),
            "Two-group structure": profile.get("has_two_group_treatment")
        }
    )

    flags = profile.get("flags", [])
    ppt_add_bullets(
        slide,
        [
            f"Dataset hash: {profile.get('dataset_hash')}",
            f"Configured columns found: {profile.get('config_validation', {}).get('all_configured_columns_found')}",
            *(flags if flags else ["No dataset-readiness flags were triggered."])
        ],
        top=3.0,
        font_size=16
    )

    # Slide 4: Method and statistics
    slide = prs.slides.add_slide(blank)
    ppt_add_title(slide, "Method Selection and Statistical Evidence", "Deterministic validation selected from the approved method registry")

    method_bullets = [
        f"Selected method: {method_decision.get('method_id')}",
        f"Reason: {method_decision.get('reason')}",
        summarize_statistical_result(validation_package)
    ]

    ppt_add_bullets(slide, method_bullets, left=0.75, top=1.4, width=5.7, font_size=15)
    ppt_add_image(slide, chart_paths.get("treatment_vs_control"), left=6.7, top=1.4, width=5.7)

    # Slide 5: Segment analysis
    if deck_style == "consulting_8_slide":
        slide = prs.slides.add_slide(blank)
        ppt_add_title(slide, "Segment Analysis", "Where the effect appears strongest or weakest")

        if segment_summary.get("available"):
            strongest = segment_summary.get("strongest_segment", {})
            weakest = segment_summary.get("weakest_segment", {})

            ppt_add_bullets(
                slide,
                [
                    f"Strongest segment: {strongest.get('segment_col')}={strongest.get('segment_value')} with {strongest.get('effect_pp'):.2f} pp lift.",
                    f"Weakest segment: {weakest.get('segment_col')}={weakest.get('segment_value')} with {weakest.get('effect_pp'):.2f} pp lift.",
                    f"Segments analyzed: {segment_summary.get('n_segments_analyzed')}"
                ],
                left=0.75,
                top=1.4,
                width=5.5,
                font_size=16
            )
            ppt_add_image(slide, chart_paths.get("segment_lift"), left=6.5, top=1.35, width=6.0)
        else:
            ppt_add_bullets(slide, ["No valid segment-level treatment effects were calculated."])

    # Slide 6: Business impact
    slide = prs.slides.add_slide(blank)
    ppt_add_title(slide, "Business Impact and Rollout Economics", "Does the evidence support the business action?")

    if impact.get("available"):
        ppt_add_metric_cards(
            slide,
            {
                "Incremental value/unit": money(impact.get("incremental_expected_value_per_unit")),
                "Financially positive": impact.get("financially_positive"),
                "Value/success": money(impact.get("business_value_per_success")),
                "Cost/success": money(impact.get("intervention_cost_per_success"))
            }
        )
        ppt_add_image(slide, chart_paths.get("segment_business_impact"), left=6.7, top=2.7, width=5.7)
    else:
        ppt_add_bullets(
            slide,
            [
                summarize_business_impact(business_review),
                "Business recommendation should not be treated as decision-ready without economic assumptions."
            ],
            font_size=17
        )

    # Slide 7: Risks and review gate
    slide = prs.slides.add_slide(blank)
    ppt_add_title(slide, "Risks, Gaps, and Human Review Gate", "Where ClaimCheck prevents overclaiming")

    review_reasons = exception_package.get("review_reasons", [])
    risk_flags = evidence_package.get("risk_flags", [])

    bullets = []
    bullets.append(f"Human review required: {exception_package.get('human_review_required')}")

    if review_reasons:
        bullets.extend(review_reasons[:5])
    elif risk_flags:
        bullets.extend(risk_flags[:5])
    else:
        bullets.append("No exception-review reasons were triggered.")

    ppt_add_bullets(slide, bullets, font_size=16)

    # Slide 8: Recommendation and audit trail
    slide = prs.slides.add_slide(blank)
    ppt_add_title(slide, "Recommendation and Audit Trail", "Final decision guidance with reproducibility record")

    ppt_add_bullets(
        slide,
        [
            f"Recommendation: {executive_brief.get('recommendation')}",
            f"Run ID: {audit_log.get('run_id')}",
            f"Timestamp UTC: {audit_log.get('timestamp_utc')}",
            f"Dataset hash: {audit_log.get('dataset_hash')}",
            f"Evidence tier: {audit_log.get('evidence_tier')}"
        ],
        font_size=16
    )

    prs.save(deck_path)
    return deck_path


# ------------------------------------------------------------
# One-click artifact bundle
# ------------------------------------------------------------

def generate_claimcheck_artifacts(
    output: Dict[str, Any],
    output_prefix: str = "claimcheck_demo",
    generate_word: bool = True,
    generate_ppt: bool = True,
    report_length: str = "full_packet",
    deck_style: str = "consulting_8_slide",
    create_zip_bundle: bool = True
) -> Dict[str, Any]:
    """
    Generate one-click ClaimCheck artifacts:
    - JSON output
    - executive brief text
    - charts
    - Word evidence report
    - PowerPoint executive deck
    - optional ZIP bundle
    """
    artifacts = {}

    saved = save_claimcheck_outputs(output, output_prefix=output_prefix)
    artifacts["json_output"] = saved["json_path"]
    artifacts["brief_text"] = saved["brief_path"]

    charts = create_claimcheck_charts(
        output=output,
        output_dir=f"{output_prefix}_charts"
    )
    artifacts["charts"] = charts

    if generate_word:
        word_path = generate_claimcheck_word_report(
            output=output,
            report_path=f"{output_prefix}_Evidence_Report.docx",
            report_length=report_length
        )
        artifacts["word_report"] = word_path

    if generate_ppt:
        ppt_path = generate_claimcheck_powerpoint_deck(
            output=output,
            deck_path=f"{output_prefix}_Executive_Deck.pptx",
            deck_style=deck_style
        )
        artifacts["powerpoint_deck"] = ppt_path

    if create_zip_bundle:
        zip_path = f"{output_prefix}_ClaimCheck_Artifact_Bundle.zip"
        files_to_zip = []

        for key in ["json_output", "brief_text", "word_report", "powerpoint_deck"]:
            if key in artifacts and artifacts[key]:
                files_to_zip.append(artifacts[key])

        for chart_path in charts.values():
            files_to_zip.append(chart_path)

        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
            for file_path in files_to_zip:
                if os.path.exists(file_path):
                    zip_file.write(file_path, arcname=os.path.basename(file_path))

        artifacts["zip_bundle"] = zip_path

    print("CLAIMCHECK ARTIFACTS GENERATED")
    print("=" * 70)
    print(json.dumps(artifacts, indent=2))

    return artifacts


print("One-Click Evidence Report and Executive Deck Generator defined.")

One-Click Evidence Report and Executive Deck Generator defined.


# 4. Evaluation Framework: Accuracy, Safety, and Business Usefulness

ClaimCheck is evaluated across four layers:

1. **Method-selection accuracy:** whether the Method Scout routes claims to the correct method family.
2. **Statistical correctness:** whether deterministic tools reproduce known lift, p-values, balance checks, and business-impact results.
3. **Governance safety:** whether unsupported causal, predictive, or decision-ready language is blocked or escalated.
4. **Business usefulness:** whether the generated brief, report, and deck help a PM or BA understand the decision, evidence, risks, and next steps.

This evaluation design aligns with the course focus on agent accuracy, safety, enterprise application, and testing methods.

## 4.1 Evaluation Harness: Labeled Test Cases for Method Selection and Safety

This evaluation harness tests whether ClaimCheck routes business claims to the correct analytical pathway and applies safe governance behavior.

The goal is not to evaluate the p-value formulas here. Those are deterministic statistical functions. Instead, this section evaluates the agentic decision layer:

1. Did ClaimCheck select the expected method family?
2. Did it classify the claim type correctly enough for business use?
3. Did it require human review when the claim was risky?
4. Did it block causal or decision-ready language when the evidence was insufficient?

This is where precision and recall fit into the project. We create a small labeled set of business claims and compare ClaimCheck's routing decisions against expected labels.

In [12]:
# ============================================================
# ClaimCheck 4.1: Evaluation Harness for Method Selection and Safety
# ============================================================

evaluation_claims = [
    {
        "case_id": "EVAL_001",
        "claim": "The campaign drove purchases and should be rolled out broadly.",
        "decision_context": "Marketing campaign rollout decision",
        "expected_method": "two_proportion_test",
        "expected_primary_claim_type": "prescriptive_recommendation",
        "expected_human_review_required": False
    },
    {
        "case_id": "EVAL_002",
        "claim": "Variant B increased average revenue compared with Variant A.",
        "decision_context": "Product experiment review",
        "expected_method": "two_sample_t_test",
        "expected_primary_claim_type": "causal_impact",
        "expected_human_review_required": False
    },
    {
        "case_id": "EVAL_003",
        "claim": "This churn score predicts which customers are likely to leave.",
        "decision_context": "Churn-risk prioritization",
        "expected_method": "predictive_driver_review",
        "expected_primary_claim_type": "predictive",
        "expected_human_review_required": True
    },
    {
        "case_id": "EVAL_004",
        "claim": "Revenue dropped after the product launch, so the launch caused the decline.",
        "decision_context": "Post-launch KPI review",
        "expected_method": "pre_post_analysis",
        "expected_primary_claim_type": "causal_impact",
        "expected_human_review_required": True
    },
    {
        "case_id": "EVAL_005",
        "claim": "Customers with higher tenure are less likely to churn.",
        "decision_context": "Observational customer analysis",
        "expected_method": "regression_adjusted_analysis",
        "expected_primary_claim_type": "predictive",
        "expected_human_review_required": True
    },
    {
        "case_id": "EVAL_006",
        "claim": "Revenue increased by 12 percent last month.",
        "decision_context": "Dashboard KPI summary",
        "expected_method": "descriptive_only",
        "expected_primary_claim_type": "descriptive",
        "expected_human_review_required": True
    }
]


def create_eval_profile_and_config(eval_case: Dict[str, Any]) -> Dict[str, Any]:
    """
    Create lightweight synthetic profiles/configs to test method routing.
    This isolates the Method Selection Engine and Validity Guard from the full dataset workflow.
    """
    case_id = eval_case["case_id"]

    base_config = {
        "dataset_name": case_id,
        "claim": eval_case["claim"],
        "decision_context": eval_case["decision_context"],
        "outcome_col": "outcome",
        "treatment_col": None,
        "group_col": None,
        "time_col": None,
        "unit_id_col": "customer_id",
        "covariates": [],
        "segment_cols": [],
        "business_value_per_success": None,
        "intervention_cost_per_success": None,
        "requested_claim_strength": "causal",
        "alpha": 0.05
    }

    base_profile = {
        "dataset_name": case_id,
        "dataset_hash": f"eval_hash_{case_id}",
        "n_rows": 1000,
        "n_columns": 5,
        "columns": ["customer_id", "outcome"],
        "column_roles": {
            "binary_cols": [],
            "numeric_cols": ["outcome"],
            "categorical_cols": [],
            "datetime_cols": []
        },
        "missing_values": {},
        "config_validation": {
            "all_configured_columns_found": True,
            "missing_configured_columns": []
        },
        "treatment_values": None,
        "has_two_group_treatment": False,
        "outcome_type": "continuous_or_numeric",
        "candidate_segments": [],
        "candidate_covariates": [],
        "flags": []
    }

    # Case-specific structures
    if case_id == "EVAL_001":
        base_config["treatment_col"] = "treatment"
        base_config["outcome_col"] = "purchase"
        base_config["business_value_per_success"] = 37.5
        base_config["intervention_cost_per_success"] = 25

        base_profile["columns"] = ["customer_id", "treatment", "purchase"]
        base_profile["outcome_type"] = "binary"
        base_profile["has_two_group_treatment"] = True
        base_profile["treatment_values"] = [0, 1]
        base_profile["column_roles"]["binary_cols"] = ["treatment", "purchase"]

    elif case_id == "EVAL_002":
        base_config["treatment_col"] = "variant"
        base_config["outcome_col"] = "revenue"

        base_profile["columns"] = ["customer_id", "variant", "revenue"]
        base_profile["outcome_type"] = "continuous_or_numeric"
        base_profile["has_two_group_treatment"] = True
        base_profile["treatment_values"] = ["A", "B"]
        base_profile["column_roles"]["numeric_cols"] = ["revenue"]
        base_profile["column_roles"]["categorical_cols"] = ["variant"]

    elif case_id == "EVAL_003":
        base_config["outcome_col"] = "churn_score"
        base_config["requested_claim_strength"] = "predictive"

        base_profile["columns"] = ["customer_id", "churn_score", "risk_segment"]
        base_profile["outcome_type"] = "continuous_or_numeric"
        base_profile["column_roles"]["numeric_cols"] = ["churn_score"]
        base_profile["column_roles"]["categorical_cols"] = ["risk_segment"]

    elif case_id == "EVAL_004":
        base_config["time_col"] = "week"
        base_config["outcome_col"] = "revenue"

        base_profile["columns"] = ["week", "revenue"]
        base_profile["outcome_type"] = "continuous_or_numeric"
        base_profile["column_roles"]["numeric_cols"] = ["revenue"]
        base_profile["column_roles"]["datetime_cols"] = ["week"]

    elif case_id == "EVAL_005":
        base_config["outcome_col"] = "churn"
        base_config["covariates"] = ["tenure", "usage", "plan_type"]

        base_profile["columns"] = ["customer_id", "churn", "tenure", "usage", "plan_type"]
        base_profile["outcome_type"] = "binary"
        base_profile["candidate_covariates"] = ["tenure", "usage", "plan_type"]
        base_profile["column_roles"]["binary_cols"] = ["churn"]
        base_profile["column_roles"]["numeric_cols"] = ["tenure", "usage"]
        base_profile["column_roles"]["categorical_cols"] = ["plan_type"]

    elif case_id == "EVAL_006":
        base_config["outcome_col"] = "revenue"

        base_profile["columns"] = ["month", "revenue"]
        base_profile["outcome_type"] = "continuous_or_numeric"
        base_profile["column_roles"]["numeric_cols"] = ["revenue"]

    return {
        "config": base_config,
        "profile": base_profile
    }


def run_method_selection_eval(evaluation_claims: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Run the evaluation cases through claim classification and method selection.
    """
    rows = []

    for eval_case in evaluation_claims:
        setup = create_eval_profile_and_config(eval_case)
        config = setup["config"]
        profile = setup["profile"]

        method_decision = select_method(profile, config)
        claim_type_package = classify_claim_type(config)

        rows.append({
            "case_id": eval_case["case_id"],
            "claim": eval_case["claim"],
            "expected_method": eval_case["expected_method"],
            "predicted_method": method_decision["method_id"],
            "method_correct": method_decision["method_id"] == eval_case["expected_method"],
            "expected_claim_type": eval_case["expected_primary_claim_type"],
            "predicted_claim_type": claim_type_package["primary_claim_type"],
            "claim_type_correct": claim_type_package["primary_claim_type"] == eval_case["expected_primary_claim_type"],
            "expected_human_review_required": eval_case["expected_human_review_required"],
            "predicted_human_review_required": method_decision["human_review_required"],
            "human_review_correct": method_decision["human_review_required"] == eval_case["expected_human_review_required"],
            "routing_reason": method_decision["reason"]
        })

    return pd.DataFrame(rows)


def calculate_precision_recall_by_method(eval_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate simple precision and recall by expected method family.
    """
    methods = sorted(set(eval_df["expected_method"]).union(set(eval_df["predicted_method"])))

    metrics = []

    for method in methods:
        true_positive = int(
            ((eval_df["expected_method"] == method) & (eval_df["predicted_method"] == method)).sum()
        )
        false_positive = int(
            ((eval_df["expected_method"] != method) & (eval_df["predicted_method"] == method)).sum()
        )
        false_negative = int(
            ((eval_df["expected_method"] == method) & (eval_df["predicted_method"] != method)).sum()
        )

        precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else None
        recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else None

        metrics.append({
            "method": method,
            "true_positive": true_positive,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "precision": precision,
            "recall": recall
        })

    return pd.DataFrame(metrics)


eval_results = run_method_selection_eval(evaluation_claims)
eval_metrics = calculate_precision_recall_by_method(eval_results)

print("Evaluation harness completed.")
print("\nRouting results:")
display(eval_results)

print("\nPrecision and recall by method:")
display(eval_metrics)

Evaluation harness completed.

Routing results:


,case_id,claim,expected_method,predicted_method,method_correct,expected_claim_type,predicted_claim_type,claim_type_correct,expected_human_review_required,predicted_human_review_required,human_review_correct,routing_reason
0,EVAL_001,The campaign drove purchases and should be rol...,two_proportion_test,two_proportion_test,True,prescriptive_recommendation,prescriptive_recommendation,True,False,False,True,Two-group treatment/control structure detected...
1,EVAL_002,Variant B increased average revenue compared w...,two_sample_t_test,two_sample_t_test,True,causal_impact,causal_impact,True,False,False,True,Two-group treatment/control structure detected...
2,EVAL_003,This churn score predicts which customers are ...,predictive_driver_review,predictive_driver_review,True,predictive,predictive,True,True,True,True,Claim appears predictive or driver-oriented ra...
3,EVAL_004,"Revenue dropped after the product launch, so t...",pre_post_analysis,pre_post_analysis,True,causal_impact,prescriptive_recommendation,False,True,True,True,Time field exists but no valid control group i...
4,EVAL_005,Customers with higher tenure are less likely t...,regression_adjusted_analysis,regression_adjusted_analysis,True,predictive,predictive,True,True,True,True,"Outcome and covariates are available, but no r..."
5,EVAL_006,Revenue increased by 12 percent last month.,descriptive_only,descriptive_only,True,descriptive,causal_impact,False,True,True,True,"No valid experimental, multi-group, pre/post, ..."



Precision and recall by method:


,method,true_positive,false_positive,false_negative,precision,recall
0,descriptive_only,1,0,0,1.0,1.0
1,pre_post_analysis,1,0,0,1.0,1.0
2,predictive_driver_review,1,0,0,1.0,1.0
3,regression_adjusted_analysis,1,0,0,1.0,1.0
4,two_proportion_test,1,0,0,1.0,1.0
5,two_sample_t_test,1,0,0,1.0,1.0


### Evaluation Result Interpretation

The first evaluation harness shows that ClaimCheck correctly routed all six labeled test cases to the expected method family. This produced perfect method-selection precision and recall on the small synthetic evaluation set.

However, the claim-type classifier made **two errors**. This is expected in a rule-based prototype because business language is ambiguous. For example, “revenue increased” can be a descriptive claim if it simply reports a KPI movement, but it can become causal if the claim attributes the increase to a campaign, product launch, or intervention.

This result is useful because it separates two evaluation findings:

1. **Method routing is currently strong on the labeled test set.**
2. **Claim-type classification requires guardrails or LLM-assisted framing in the CrewAI version.**

In production, ClaimCheck would use an LLM Claim Framer to interpret the claim in context, while deterministic guardrails would still control method selection, statistical execution, and language safety.

## 4.2 Claim-Type and Language Safety Evaluation

The second evaluation layer tests whether ClaimCheck prevents unsupported business language.

This matters because the most common enterprise analytics risk is not only choosing the wrong statistical test. It is allowing unsupported conclusions to enter a slide, dashboard, memo, or executive recommendation.

This test evaluates whether the Language Guardrail blocks causal or decision-ready language when the evidence tier is descriptive, predictive, or suggestive.

In [13]:
# ============================================================
# Claim-Type and Language Safety Evaluation
# ============================================================

language_safety_tests = [
    {
        "case_id": "LANG_001",
        "text": "The campaign caused purchases to increase and should roll out to all customers.",
        "evidence_tier": "Descriptive",
        "expected_violation": True
    },
    {
        "case_id": "LANG_002",
        "text": "The churn model predicts higher risk for low-tenure customers.",
        "evidence_tier": "Predictive",
        "expected_violation": False
    },
    {
        "case_id": "LANG_003",
        "text": "The dashboard pattern proves that the launch drove retention.",
        "evidence_tier": "Suggestive",
        "expected_violation": True
    },
    {
        "case_id": "LANG_004",
        "text": "The experiment showed a statistically significant treatment effect.",
        "evidence_tier": "Causal",
        "expected_violation": False
    },
    {
        "case_id": "LANG_005",
        "text": "The result is risk-free and guarantees profitable growth.",
        "evidence_tier": "Decision-Ready",
        "expected_violation": True
    }
]


def run_language_safety_eval(language_tests: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Evaluate whether the language guardrail correctly flags unsupported wording.
    """
    rows = []

    for test in language_tests:
        scan = scan_for_blocked_language(
            text=test["text"],
            evidence_tier=test["evidence_tier"]
        )

        rows.append({
            "case_id": test["case_id"],
            "evidence_tier": test["evidence_tier"],
            "text": test["text"],
            "expected_violation": test["expected_violation"],
            "predicted_violation": scan["language_violation"],
            "correct": scan["language_violation"] == test["expected_violation"],
            "blocked_terms_found": scan["blocked_terms_found"]
        })

    return pd.DataFrame(rows)


language_eval_results = run_language_safety_eval(language_safety_tests)

language_accuracy = language_eval_results["correct"].mean()

print("Language safety evaluation completed.")
print(f"Language safety accuracy: {language_accuracy:.2%}")

display(language_eval_results)

Language safety evaluation completed.
Language safety accuracy: 80.00%


,case_id,evidence_tier,text,expected_violation,predicted_violation,correct,blocked_terms_found
0,LANG_001,Descriptive,The campaign caused purchases to increase and ...,True,True,True,"[caused, should roll out]"
1,LANG_002,Predictive,The churn model predicts higher risk for low-t...,False,False,True,[]
2,LANG_003,Suggestive,The dashboard pattern proves that the launch d...,True,False,False,[]
3,LANG_004,Causal,The experiment showed a statistically signific...,False,False,True,[]
4,LANG_005,Decision-Ready,The result is risk-free and guarantees profita...,True,True,True,"[guarantees, risk-free]"


In [14]:
# ============================================================
# 4.3: Evaluation Summary
# ============================================================

evaluation_summary = {
    "method_selection_accuracy": eval_results["method_correct"].mean(),
    "claim_type_accuracy": eval_results["claim_type_correct"].mean(),
    "human_review_routing_accuracy": eval_results["human_review_correct"].mean(),
    "language_safety_accuracy": language_eval_results["correct"].mean()
}

evaluation_summary_df = pd.DataFrame(
    list(evaluation_summary.items()),
    columns=["evaluation_metric", "score"]
)

display(evaluation_summary_df)

,evaluation_metric,score
0,method_selection_accuracy,1.000000
1,claim_type_accuracy,0.666667
2,human_review_routing_accuracy,1.000000
3,language_safety_accuracy,0.800000


### 4.2.1 Guardrail Improvement Based on Evaluation Findings

The language-safety evaluation identified one clear gap. Under the Suggestive evidence tier, ClaimCheck failed to flag the sentence:

> "The dashboard pattern proves that the launch drove retention."

This happened because the deterministic blocked-language list included "proved" but did not include the variation "proves", and the Suggestive tier did not explicitly block "drove".

This is an example of why evaluation is necessary. The guardrail logic can be improved directly from failed test cases. In a production version, this rule set would be expanded using a larger library of reviewed claims and potentially supported by an LLM Claim Framer, but deterministic blocked-language enforcement would remain important for safety.

In [16]:
# ============================================================
# ClaimCheck 4.2.1: Guardrail Improvement Based on Evaluation Findings
# ============================================================

# Extend blocked terms based on failed language-safety test.
BANNED_LANGUAGE_RULES["Suggestive"]["blocked_terms"] = list(set(
    BANNED_LANGUAGE_RULES["Suggestive"]["blocked_terms"]
    + [
        "prove",
        "proves",
        "proved",
        "proven",
        "drove",
        "driven by",
        "caused",
        "causes",
        "causal impact",
        "resulted in",
        "led to"
    ]
))

# Add rewrite mappings for additional variants.
LANGUAGE_REWRITE_MAP.update({
    "prove": "suggest",
    "proves": "suggests",
    "proved": "provided evidence that",
    "proven": "supported by evidence",
    "drove": "is associated with",
    "driven by": "associated with",
    "causes": "is associated with"
})

# Rerun the language safety evaluation after updating the guardrail.
language_eval_results_v2 = run_language_safety_eval(language_safety_tests)
language_accuracy_v2 = language_eval_results_v2["correct"].mean()

print("Updated language safety evaluation completed.")
print(f"Updated language safety accuracy: {language_accuracy_v2:.2%}")

display(language_eval_results_v2)

Updated language safety evaluation completed.
Updated language safety accuracy: 100.00%


,case_id,evidence_tier,text,expected_violation,predicted_violation,correct,blocked_terms_found
0,LANG_001,Descriptive,The campaign caused purchases to increase and ...,True,True,True,"[caused, should roll out]"
1,LANG_002,Predictive,The churn model predicts higher risk for low-t...,False,False,True,[]
2,LANG_003,Suggestive,The dashboard pattern proves that the launch d...,True,True,True,"[prove, proves, drove]"
3,LANG_004,Causal,The experiment showed a statistically signific...,False,False,True,[]
4,LANG_005,Decision-Ready,The result is risk-free and guarantees profita...,True,True,True,"[guarantees, risk-free]"


In [17]:
# ============================================================
# ClaimCheck 4.2.2: Updated Evaluation Summary
# ============================================================

evaluation_summary_v2 = {
    "method_selection_accuracy": eval_results["method_correct"].mean(),
    "claim_type_accuracy": eval_results["claim_type_correct"].mean(),
    "human_review_routing_accuracy": eval_results["human_review_correct"].mean(),
    "language_safety_accuracy_before_guardrail_update": language_eval_results["correct"].mean(),
    "language_safety_accuracy_after_guardrail_update": language_eval_results_v2["correct"].mean()
}

evaluation_summary_v2_df = pd.DataFrame(
    list(evaluation_summary_v2.items()),
    columns=["evaluation_metric", "score"]
)

display(evaluation_summary_v2_df)

,evaluation_metric,score
0,method_selection_accuracy,1.000000
1,claim_type_accuracy,0.666667
2,human_review_routing_accuracy,1.000000
3,language_safety_accuracy_before_guardrail_update,0.800000
4,language_safety_accuracy_after_guardrail_update,1.000000


### 4.2.3 Evaluation Update Interpretation

The updated evaluation summary shows that ClaimCheck's deterministic method-selection and human-review routing layers performed strongly on the labeled test set.

- **Method selection accuracy:** 100%
- **Human-review routing accuracy:** 100%
- **Language safety accuracy:** improved from 80% to 100% after updating the Suggestive-tier guardrail rules

This demonstrates an important agentic AI review cycle:

1. Run evaluation.
2. Identify a failed safety case.
3. Update the deterministic guardrail.
4. Rerun the evaluation.
5. Record the improvement.

The weakest layer is currently **claim-type classification**, which achieved 66.7% accuracy. This is expected because business language is ambiguous. For example, "revenue increased" may be descriptive in one context but causal in another. In the CrewAI version, this task will be handled by the Claim Framer agent using an LLM, while deterministic tools continue to control method routing, statistical validation, language enforcement, and audit logging.

This evaluation supports the design choice behind ClaimCheck: use LLM agents for interpretation and communication, but use deterministic tools for calculation, safety, and governance.

# 5. Evaluation Case 1: Treatment-Control Campaign Review

This case tests ClaimCheck on a treatment-control campaign experiment. The submitted stakeholder claim is:

> "The campaign drove purchases and should be rolled out broadly."

This is intentionally written as a realistic product or marketing claim. It combines a causal claim with a prescriptive recommendation.

GameFun is used as the first benchmark because it has a treatment/control design and a prior human analysis to compare against. It is not hard-coded into the ClaimCheck engine. ClaimCheck receives the dataset, claim, column configuration, and business assumptions, then applies the same reusable review workflow used for any comparable treatment-control business case.

**Expected benchmark pattern:** treatment and control should be balanced on observable covariates; purchase lift should be positive overall; segment heterogeneity should matter; broad rollout should be questioned if campaign economics are unfavorable.

ClaimCheck must evaluate whether the available dataset supports the submitted claim by checking:

1. whether the dataset is sufficient;
2. whether treatment and control groups are valid;
3. whether the selected method is appropriate;
4. whether the treatment effect is statistically supported;
5. whether the recommendation is commercially justified;
6. whether the claim should be auto-cleared or routed to human review;
7. whether the final output should allow causal and decision-ready language.

This case is important because a campaign can increase purchase probability but still fail the business-impact test if the treatment cost is too high. ClaimCheck should therefore separate statistical lift from rollout readiness.

## 5.1 Load and Inspect the Case 1 Dataset

This cell loads the treatment-control campaign dataset and inspects its structure before ClaimCheck is configured.

The purpose of this step is to avoid hard-coding dataset assumptions. ClaimCheck should first inspect the available columns, sample size, missingness, and possible treatment, outcome, covariate, and segment fields. The analyst then maps the relevant columns into the ClaimCheck configuration.

In [18]:
# ============================================================
# ClaimCheck 5.1: Load and Inspect Case 1 Dataset
# ============================================================

from google.colab import files

uploaded = files.upload()

case1_file_name = next(iter(uploaded.keys()))
print(f"Uploaded file: {case1_file_name}")

if case1_file_name.lower().endswith(".csv"):
    case1_df = pd.read_csv(case1_file_name)
elif case1_file_name.lower().endswith((".xlsx", ".xls")):
    case1_df = pd.read_excel(case1_file_name)
else:
    raise ValueError("Please upload a CSV or Excel file.")

print("Case 1 dataset loaded.")
print(f"Rows: {case1_df.shape[0]}")
print(f"Columns: {case1_df.shape[1]}")

display(case1_df.head())
print("\nColumn names:")
print(case1_df.columns.tolist())

print("\nMissing values:")
display(case1_df.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_values"}))

print("\nColumn data types:")
display(case1_df.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))

Saving GameFun (1).xlsx to GameFun (1).xlsx
Uploaded file: GameFun (1).xlsx
Case 1 dataset loaded.
Rows: 40048
Columns: 8


,id,test,purchase,site,impressions,income,gender,gamer
0,1956,0,0,site1,0,100,1,0
1,45821,1,0,site1,20,70,1,0
2,59690,1,0,site1,22,100,1,0
3,18851,0,0,site1,13,90,1,0
4,60647,1,0,site1,12,60,1,0



Column names:
['id', 'test', 'purchase', 'site', 'impressions', 'income', 'gender', 'gamer']

Missing values:


,column,missing_values
0,id,0
1,test,0
2,purchase,0
3,site,0
4,impressions,0
5,income,0
6,gender,0
7,gamer,0



Column data types:


,column,dtype
0,id,int64
1,test,int64
2,purchase,int64
3,site,object
4,impressions,int64
5,income,int64
6,gender,int64
7,gamer,int64


## 5.2 Configure ClaimCheck for Case 1

The dataset contains a clean treatment-control structure:

- `test`: treatment indicator, where 0 is control and 1 is treatment
- `purchase`: binary outcome
- `income`, `gender`, and `gamer`: observable covariates for balance checks
- `gender` and `gamer`: business segments for heterogeneous treatment-effect review

The business assumption used for this benchmark follows the campaign economics: each successful purchase is worth USD 37.50, while the promotional treatment has a USD 25.00 cost per successful purchase. This allows ClaimCheck to separate statistical lift from business rollout readiness.

In [19]:
# ============================================================
# ClaimCheck 5.2: Configure ClaimCheck for Case 1
# ============================================================

case1_config = {
    "dataset_name": "GameFun treatment-control campaign benchmark",
    "claim": "The campaign drove purchases and should be rolled out broadly.",
    "decision_context": "Evaluate whether the promotional campaign should be approved for broad rollout or targeted use.",
    "outcome_col": "purchase",
    "treatment_col": "test",
    "group_col": None,
    "time_col": None,
    "unit_id_col": "id",
    "covariates": ["income", "gender", "gamer"],
    "segment_cols": ["gender", "gamer"],
    "business_value_per_success": 37.5,
    "intervention_cost_per_success": 25.0,
    "requested_claim_strength": "causal",
    "alpha": 0.05
}

print("Case 1 configuration created.")
print(json.dumps(case1_config, indent=2))

Case 1 configuration created.
{
  "dataset_name": "GameFun treatment-control campaign benchmark",
  "claim": "The campaign drove purchases and should be rolled out broadly.",
  "decision_context": "Evaluate whether the promotional campaign should be approved for broad rollout or targeted use.",
  "outcome_col": "purchase",
  "treatment_col": "test",
  "group_col": null,
  "time_col": null,
  "unit_id_col": "id",
  "covariates": [
    "income",
    "gender",
    "gamer"
  ],
  "segment_cols": [
    "gender",
    "gamer"
  ],
  "business_value_per_success": 37.5,
  "intervention_cost_per_success": 25.0,
  "requested_claim_strength": "causal",
  "alpha": 0.05
}


## 5.3 Run ClaimCheck on Case 1

This cell runs the full ClaimCheck workflow on the treatment-control campaign dataset.

The workflow includes dataset profiling, method selection, statistical validation, balance checks, segment review, business-impact calculation, evidence-tier assignment, language guardrails, human-review routing, executive brief generation, and audit logging.

In [20]:
# ============================================================
# 5.3: Run Full ClaimCheck Workflow on Case 1
# ============================================================

case1_output = run_claimcheck(
    df=case1_df,
    config=case1_config,
    verbose=True
)

DATASET PROFILE
Dataset: GameFun treatment-control campaign benchmark
Dataset hash: d387bd129ff8a2f9
Rows: 40,048
Columns: 8
Outcome type: binary
Two-group treatment structure: True

Detected column roles:
{
  "binary_cols": [
    "test",
    "purchase",
    "gender",
    "gamer"
  ],
  "numeric_cols": [
    "id",
    "test",
    "purchase",
    "impressions",
    "income",
    "gender",
    "gamer"
  ],
  "categorical_cols": [
    "test",
    "purchase",
    "site",
    "income",
    "gender",
    "gamer"
  ],
  "datetime_cols": []
}

Config validation:
{
  "all_configured_columns_found": true,
  "missing_configured_columns": []
}

Missing values: none detected

Profile flags: none


METHOD SELECTION DECISION
Selected method family: two_proportion_test
Prototype status: implemented
Auto-runnable in prototype: True
Human review required: False
Causal language allowed at this stage: True
Reason: Two-group treatment/control structure detected with a binary outcome.

Registry metadata:
Bu

/tmp/ipykernel_746/2642097528.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = f"claimcheck_{dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
/tmp/ipykernel_746/2642097528.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": dt.datetime.utcnow().isoformat(),


,segment_col,segment_value,segment_n,control_rate,treatment_rate,effect_pp,p_value
0,gender,0,14118,0.034442,0.080945,4.650289,5.274714e-24
1,gender,1,25930,0.037176,0.074575,3.739947,1.145194e-29
2,gamer,0,15960,0.037387,0.035092,-0.229468,4.751695e-01
3,gamer,1,24088,0.035436,0.104487,6.905098,4.424085e-70
4,gender+gamer,0+0,5595,0.038050,0.036289,-0.176065,7.484635e-01
5,gender+gamer,0+1,8523,0.032041,0.110092,7.805060,1.583582e-31
6,gender+gamer,1+0,10365,0.037025,0.034450,-0.257538,5.156773e-01
7,gender+gamer,1+1,15565,0.037275,0.101404,6.412899,1.196271e-40




VALIDITY GUARD OUTPUT

Claim type and evidence tier:
{
  "claim_type": {
    "primary_claim_type": "prescriptive_recommendation",
    "detected_claim_types": [
      "prescriptive_recommendation",
      "causal_impact"
    ]
  },
  "evidence_tier": "Causal",
  "requested_claim_strength": "causal",
  "risk_flags": [
    "Overall business impact is negative despite statistical lift."
  ],
  "reasons": [
    "Implemented treatment/control validation ran successfully and the result is statistically significant.",
    "Configured covariates appear balanced between treatment and control groups.",
    "Statistical improvement does not justify broad rollout because expected value is negative."
  ],
  "causal_language_allowed": true,
  "decision_language_allowed": false
}

Exception routing:
{
  "human_review_required": true,
  "review_type": "exception_review",
  "review_reasons": [
    "Overall business impact is negative despite statistical lift.",
    "Claim asks for a business recommenda

### 5.3.1 Business Evidence Extraction for LLM Briefing

The first ClaimCheck run produced the correct statistical and business evidence, but some raw outputs are not stakeholder-friendly. For example, `gender+gamer = 0+1` is technically correct but not useful for a business audience.

This section does not hard-code the analyst recommendation. Instead, it creates a structured business evidence packet that an LLM Brief Builder agent can use to generate an executive summary, report narrative, and PowerPoint storyline.

The deterministic layer remains responsible for calculations and governance. The LLM layer is responsible for communication.

In [24]:
# ============================================================
# 5.3.1: Business Evidence Packet for LLM Brief Builder
# ============================================================

case1_label_maps = {
    "test": {
        0: "Control",
        1: "Treatment"
    },
    "gender": {
        0: "Female",
        1: "Male"
    },
    "gamer": {
        0: "Non-gamer",
        1: "Gamer"
    },
    "purchase": {
        0: "No purchase",
        1: "Purchase"
    }
}


def label_value(column: str, value: Any, label_maps: Dict[str, Dict[Any, str]]) -> str:
    """
    Convert coded dataset values into human-readable labels.
    """
    if column in label_maps:
        return label_maps[column].get(value, str(value))
    return str(value)


def label_segment(segment_col: str, segment_value: Any, label_maps: Dict[str, Dict[Any, str]]) -> str:
    """
    Convert raw segment labels such as gender+gamer = 0+1 into Female Gamer.
    """
    segment_col = str(segment_col)
    segment_value = str(segment_value)

    if "+" not in segment_col:
        clean_value = int(segment_value) if segment_value.isdigit() else segment_value
        return label_value(segment_col, clean_value, label_maps)

    cols = segment_col.split("+")
    vals = segment_value.split("+")

    readable_parts = []

    for col, val in zip(cols, vals):
        clean_val = int(val) if str(val).isdigit() else val
        readable_parts.append(label_value(col, clean_val, label_maps))

    return " ".join(readable_parts)


def build_business_evidence_packet(
    output: Dict[str, Any],
    label_maps: Dict[str, Dict[Any, str]]
) -> Dict[str, Any]:
    """
    Build a structured evidence packet for the LLM Brief Builder.

    This packet contains validated facts only. It does not hard-code the final business narrative.
    """
    stat = output["validation_package"]["statistical_result"]
    impact = output["business_review"]["overall_business_impact"]
    segment_results = output["business_review"]["segment_results"]
    validity = output["validity_output"]
    evidence = validity["evidence_package"]
    exception = validity["exception_package"]

    readable_segments = []

    for row in segment_results:
        if "error" in row:
            continue

        segment_impact = row.get("business_impact", {})

        readable_segments.append({
            "segment": label_segment(row.get("segment_col"), row.get("segment_value"), label_maps),
            "raw_segment_col": row.get("segment_col"),
            "raw_segment_value": row.get("segment_value"),
            "segment_n": row.get("segment_n"),
            "control_purchase_rate": row.get("control_rate"),
            "treatment_purchase_rate": row.get("treatment_rate"),
            "lift_percentage_points": row.get("effect_pp"),
            "p_value": row.get("p_value"),
            "incremental_expected_value_per_customer": segment_impact.get("incremental_expected_value_per_unit")
                if segment_impact.get("available") else None,
            "financially_positive": segment_impact.get("financially_positive")
                if segment_impact.get("available") else None
        })

    positive_segments = [
        row for row in readable_segments
        if row["financially_positive"] is True
    ]

    positive_segments = sorted(
        positive_segments,
        key=lambda x: x["incremental_expected_value_per_customer"],
        reverse=True
    )

    strongest_lift_segment = None
    if readable_segments:
        strongest_lift_segment = sorted(
            readable_segments,
            key=lambda x: x["lift_percentage_points"],
            reverse=True
        )[0]

    evidence_packet = {
        "submitted_claim": output["audit_log"]["original_claim"],
        "decision_context": output["audit_log"]["decision_context"],

        "overall_statistical_result": {
            "control_purchase_rate": stat.get("control_rate"),
            "treatment_purchase_rate": stat.get("treatment_rate"),
            "lift_percentage_points": stat.get("effect_pp"),
            "p_value": stat.get("p_value"),
            "confidence_interval_95_pp": [
                stat.get("ci_95_low_pp"),
                stat.get("ci_95_high_pp")
            ],
            "statistically_significant": stat.get("statistically_significant")
        },

        "overall_business_result": {
            "control_expected_value_per_customer": impact.get("control_expected_value_per_unit"),
            "treatment_expected_value_per_customer": impact.get("treatment_expected_value_per_unit"),
            "incremental_expected_value_per_customer": impact.get("incremental_expected_value_per_unit"),
            "financially_positive": impact.get("financially_positive")
        },

        "segment_results": readable_segments,
        "financially_positive_segments": positive_segments,
        "strongest_lift_segment": strongest_lift_segment,

        "governance_result": {
            "claim_type": evidence["claim_type"]["primary_claim_type"],
            "evidence_tier": evidence["evidence_tier"],
            "causal_language_allowed": evidence["causal_language_allowed"],
            "decision_language_allowed": evidence["decision_language_allowed"],
            "human_review_required": exception["human_review_required"],
            "review_reasons": exception["review_reasons"]
        },

        "llm_brief_builder_instruction": (
            "Generate a business analyst interpretation using only the facts in this packet. "
            "Do not invent new statistics. Clearly separate statistical lift from business rollout readiness. "
            "Do not approve broad rollout if overall business impact is negative. "
            "If financially positive segments exist, mention them as targeted opportunities subject to review. "
            "Respect the evidence tier and language permissions."
        )
    }

    return evidence_packet


case1_business_evidence_packet = build_business_evidence_packet(
    output=case1_output,
    label_maps=case1_label_maps
)

print(json.dumps(case1_business_evidence_packet, indent=2))

{
  "submitted_claim": "The campaign drove purchases and should be rolled out broadly.",
  "decision_context": "Evaluate whether the promotional campaign should be approved for broad rollout or targeted use.",
  "overall_statistical_result": {
    "control_purchase_rate": 0.03621309693066823,
    "treatment_purchase_rate": 0.07682175785838881,
    "lift_percentage_points": 4.0608660927720575,
    "p_value": 1.2262726811132347e-51,
    "confidence_interval_95_pp": [
      3.6035685006417206,
      4.518163684902396
    ],
    "statistically_significant": true
  },
  "overall_business_result": {
    "control_expected_value_per_customer": 1.3579911349000586,
    "treatment_expected_value_per_customer": 0.9602719732298601,
    "incremental_expected_value_per_customer": -0.3977191616701985,
    "financially_positive": false
  },
  "segment_results": [
    {
      "segment": "Female",
      "raw_segment_col": "gender",
      "raw_segment_value": 0,
      "segment_n": 14118,
      "control_pu

## 5.4 Reusable Brief Builder Prompt Template

This section defines a reusable Brief Builder prompt template.

The prompt is not specific to GameFun. It is designed to accept any ClaimCheck evidence packet and instruct an LLM agent to generate a business-facing analyst brief using only validated evidence.

This separates the architecture into two layers:

1. **Deterministic evidence layer:** calculates statistics, business impact, evidence tier, review status, and audit trail.
2. **LLM communication layer:** converts the validated evidence packet into clear business language, report narrative, and slide storyline.

The same template can be reused across treatment-control cases, predictive review cases, descriptive KPI cases, or unsupported causal claims. Only the evidence packet changes.

In [27]:
# ============================================================
# ClaimCheck 5.4: Reusable Brief Builder Prompt Template
# ============================================================

def build_brief_builder_prompt(
    evidence_packet: Dict[str, Any],
    audience: str = "executive",
    output_format: str = "analyst_brief"
) -> str:
    """
    Build a reusable prompt for the Brief Builder agent.

    This function is case-agnostic. It can be used for any ClaimCheck evidence packet.
    """
    prompt = f"""
You are the ClaimCheck Brief Builder Agent.

Audience:
{audience}

Output format:
{output_format}

Your task:
Convert the validated ClaimCheck evidence packet into a clear business-facing interpretation.

Important rules:
1. Use only the facts provided in the evidence packet.
2. Do not invent statistics, sample sizes, p-values, confidence intervals, segment names, or business-impact values.
3. Clearly separate statistical evidence from business decision readiness.
4. Respect the evidence tier.
5. Respect language permissions.
6. If decision language is not allowed, do not write as if the recommendation is fully approved.
7. If human review is required, explain why.
8. If financially positive segments exist, describe them as targeted opportunities, not automatic rollout approval.
9. If overall business impact is negative, do not recommend broad rollout.
10. Write in a concise, professional business analyst style.

Required sections:
- Executive answer
- Evidence summary
- Segment or driver insight
- Business impact
- Governance and review status
- Recommended next step

Validated evidence packet:
{json.dumps(evidence_packet, indent=2)}
"""
    return prompt.strip()


case1_brief_builder_prompt = build_brief_builder_prompt(
    evidence_packet=case1_business_evidence_packet,
    audience="product and marketing leadership",
    output_format="analyst_brief"
)

print(case1_brief_builder_prompt[:2500])
print("\n--- Prompt truncated for display. Full prompt stored in case1_brief_builder_prompt. ---")

You are the ClaimCheck Brief Builder Agent.

Audience:
product and marketing leadership

Output format:
analyst_brief

Your task:
Convert the validated ClaimCheck evidence packet into a clear business-facing interpretation.

Important rules:
1. Use only the facts provided in the evidence packet.
2. Do not invent statistics, sample sizes, p-values, confidence intervals, segment names, or business-impact values.
3. Clearly separate statistical evidence from business decision readiness.
4. Respect the evidence tier.
5. Respect language permissions.
6. If decision language is not allowed, do not write as if the recommendation is fully approved.
7. If human review is required, explain why.
8. If financially positive segments exist, describe them as targeted opportunities, not automatic rollout approval.
9. If overall business impact is negative, do not recommend broad rollout.
10. Write in a concise, professional business analyst style.

Required sections:
- Executive answer
- Evidence summ

### 5.4.1 Lightweight Run Memory for Rework and Auditability

A real agentic workflow needs memory. If a stakeholder revises the claim, changes the business assumption, or asks for a more cautious recommendation, ClaimCheck should retain the previous run and compare it against the new run.

In this Colab prototype, memory is represented as a lightweight Python run store. In the CrewAI production architecture, this would become a combination of short-term task context, audit logs, and long-term retrieval over prior reviewed claims.

In [28]:
# ============================================================
# ClaimCheck 5.4.1: Lightweight Run Memory for Rework and Auditability
# ============================================================

claimcheck_memory = {
    "runs": []
}


def save_run_to_memory(
    memory: Dict[str, Any],
    case_name: str,
    output: Dict[str, Any],
    evidence_packet: Dict[str, Any],
    brief_prompt: str
) -> Dict[str, Any]:
    """
    Save a ClaimCheck run into lightweight notebook memory.
    """
    memory_record = {
        "case_name": case_name,
        "run_id": output["audit_log"]["run_id"],
        "timestamp_utc": output["audit_log"]["timestamp_utc"],
        "dataset_hash": output["audit_log"]["dataset_hash"],
        "claim": output["audit_log"]["original_claim"],
        "selected_method": output["audit_log"]["selected_method"],
        "evidence_tier": output["audit_log"]["evidence_tier"],
        "human_review_required": output["audit_log"]["human_review_required"],
        "recommendation": output["audit_log"]["recommendation"],
        "evidence_packet": evidence_packet,
        "brief_prompt": brief_prompt
    }

    memory["runs"].append(memory_record)
    return memory_record


case1_memory_record = save_run_to_memory(
    memory=claimcheck_memory,
    case_name="Case 1: GameFun treatment-control benchmark",
    output=case1_output,
    evidence_packet=case1_business_evidence_packet,
    brief_prompt=case1_brief_builder_prompt
)

print("Run saved to ClaimCheck memory.")
print(json.dumps({
    "case_name": case1_memory_record["case_name"],
    "run_id": case1_memory_record["run_id"],
    "claim": case1_memory_record["claim"],
    "evidence_tier": case1_memory_record["evidence_tier"],
    "human_review_required": case1_memory_record["human_review_required"]
}, indent=2))

Run saved to ClaimCheck memory.
{
  "case_name": "Case 1: GameFun treatment-control benchmark",
  "run_id": "claimcheck_20260723_180248",
  "claim": "The campaign drove purchases and should be rolled out broadly.",
  "evidence_tier": "Causal",
  "human_review_required": true
}


## 5.5 Generate Case 1 Report and PowerPoint Artifacts

This section generates the shareable Case 1 artifact bundle.

The bundle includes:

1. JSON ClaimCheck output for reproducibility.
2. Business evidence packet for the LLM Brief Builder.
3. Brief Builder prompt for agentic interpretation.
4. Word evidence report.
5. PowerPoint executive deck.
6. ZIP bundle containing the generated files.

In this Colab prototype, the report and deck are generated from deterministic ClaimCheck outputs. In the full CrewAI version, the Brief Builder agent will convert the evidence packet into a more polished analyst narrative before the same report and deck generator creates the final artifacts.

In [29]:
# ============================================================
# ClaimCheck 5.5: Generate Case 1 Report and PowerPoint Artifacts
# ============================================================

# Generate the standard ClaimCheck artifact bundle from the deterministic output.
case1_artifacts = generate_claimcheck_artifacts(
    output=case1_output,
    output_prefix="claimcheck_case1_gamefun",
    generate_word=True,
    generate_ppt=True,
    report_length="full_packet",
    deck_style="consulting_8_slide",
    create_zip_bundle=True
)

# Save the business evidence packet separately.
case1_evidence_packet_path = "claimcheck_case1_gamefun_business_evidence_packet.json"

with open(case1_evidence_packet_path, "w") as f:
    json.dump(case1_business_evidence_packet, f, indent=2)

case1_artifacts["business_evidence_packet"] = case1_evidence_packet_path

# Save the reusable LLM Brief Builder prompt separately.
case1_prompt_path = "claimcheck_case1_gamefun_brief_builder_prompt.txt"

with open(case1_prompt_path, "w") as f:
    f.write(case1_brief_builder_prompt)

case1_artifacts["brief_builder_prompt"] = case1_prompt_path

# Rebuild ZIP bundle to include evidence packet and prompt.
case1_zip_path = "claimcheck_case1_gamefun_complete_artifact_bundle.zip"

files_to_zip = []

for key, value in case1_artifacts.items():
    if isinstance(value, str) and os.path.exists(value):
        files_to_zip.append(value)

if "charts" in case1_artifacts:
    for chart_path in case1_artifacts["charts"].values():
        if os.path.exists(chart_path):
            files_to_zip.append(chart_path)

with zipfile.ZipFile(case1_zip_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
    for file_path in files_to_zip:
        zip_file.write(file_path, arcname=os.path.basename(file_path))

case1_artifacts["complete_zip_bundle"] = case1_zip_path

print("Case 1 artifacts generated.")
print("=" * 70)
print(json.dumps(case1_artifacts, indent=2))

CLAIMCHECK ARTIFACTS GENERATED
{
  "json_output": "claimcheck_case1_gamefun.json",
  "brief_text": "claimcheck_case1_gamefun_executive_brief.txt",
  "charts": {
    "treatment_vs_control": "claimcheck_case1_gamefun_charts/treatment_vs_control.png",
    "segment_lift": "claimcheck_case1_gamefun_charts/segment_lift.png",
    "segment_business_impact": "claimcheck_case1_gamefun_charts/segment_business_impact.png"
  },
  "word_report": "claimcheck_case1_gamefun_Evidence_Report.docx",
  "powerpoint_deck": "claimcheck_case1_gamefun_Executive_Deck.pptx",
  "zip_bundle": "claimcheck_case1_gamefun_ClaimCheck_Artifact_Bundle.zip"
}
Case 1 artifacts generated.
{
  "json_output": "claimcheck_case1_gamefun.json",
  "brief_text": "claimcheck_case1_gamefun_executive_brief.txt",
  "charts": {
    "treatment_vs_control": "claimcheck_case1_gamefun_charts/treatment_vs_control.png",
    "segment_lift": "claimcheck_case1_gamefun_charts/segment_lift.png",
    "segment_business_impact": "claimcheck_case1_gam

In [30]:
# ============================================================
# Optional: Download Case 1 Artifact Bundle
# ============================================================

from google.colab import files

files.download(case1_artifacts["complete_zip_bundle"])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5.6 Case 1 Benchmark Comparison and Learning

This section compares ClaimCheck's output against the expected benchmark pattern for the GameFun treatment-control case.

The goal is not only to show that ClaimCheck produced an output, but to evaluate whether it recovered the same business logic a human analyst would expect:

1. treatment and control are balanced;
2. overall purchase lift is positive and statistically significant;
3. broad rollout is not automatically justified because campaign economics are unfavorable;
4. segment heterogeneity matters;
5. targeted review is more appropriate than broad rollout.

This closes the first evaluation case by showing that ClaimCheck can reproduce the core human analysis pattern while adding evidence-tiering, language guardrails, human-review routing, and artifact generation.

In [31]:
# ============================================================
# ClaimCheck 5.6: Case 1 Benchmark Comparison
# ============================================================

def evaluate_case1_benchmark(output: Dict[str, Any], evidence_packet: Dict[str, Any]) -> pd.DataFrame:
    """
    Compare ClaimCheck output against expected benchmark patterns for the GameFun case.
    """
    balance = output["validation_package"]["balance_result"]
    stat = evidence_packet["overall_statistical_result"]
    business = evidence_packet["overall_business_result"]
    governance = evidence_packet["governance_result"]
    positive_segments = evidence_packet["financially_positive_segments"]

    benchmark_checks = [
        {
            "benchmark_expectation": "Treatment and control are balanced on observable covariates",
            "claimcheck_result": balance.get("balanced"),
            "passed": balance.get("balanced") is True
        },
        {
            "benchmark_expectation": "Overall purchase lift is positive",
            "claimcheck_result": stat.get("lift_percentage_points"),
            "passed": stat.get("lift_percentage_points", 0) > 0
        },
        {
            "benchmark_expectation": "Overall purchase lift is statistically significant",
            "claimcheck_result": stat.get("statistically_significant"),
            "passed": stat.get("statistically_significant") is True
        },
        {
            "benchmark_expectation": "Broad rollout should be questioned if economics are unfavorable",
            "claimcheck_result": business.get("financially_positive"),
            "passed": business.get("financially_positive") is False
        },
        {
            "benchmark_expectation": "Segment heterogeneity should matter",
            "claimcheck_result": len(evidence_packet.get("segment_results", [])),
            "passed": len(evidence_packet.get("segment_results", [])) > 0
        },
        {
            "benchmark_expectation": "At least one targeted opportunity should be identified if financially positive segments exist",
            "claimcheck_result": [row["segment"] for row in positive_segments],
            "passed": len(positive_segments) > 0
        },
        {
            "benchmark_expectation": "Decision-ready broad rollout language should not be allowed",
            "claimcheck_result": governance.get("decision_language_allowed"),
            "passed": governance.get("decision_language_allowed") is False
        },
        {
            "benchmark_expectation": "Human review should be required for negative overall economics with positive segment opportunity",
            "claimcheck_result": governance.get("human_review_required"),
            "passed": governance.get("human_review_required") is True
        }
    ]

    return pd.DataFrame(benchmark_checks)


case1_benchmark_results = evaluate_case1_benchmark(
    output=case1_output,
    evidence_packet=case1_business_evidence_packet
)

display(case1_benchmark_results)

case1_benchmark_pass_rate = case1_benchmark_results["passed"].mean()

print(f"Case 1 benchmark pass rate: {case1_benchmark_pass_rate:.2%}")

,benchmark_expectation,claimcheck_result,passed
0,Treatment and control are balanced on observab...,True,True
1,Overall purchase lift is positive,4.060866,True
2,Overall purchase lift is statistically signifi...,True,True
3,Broad rollout should be questioned if economic...,False,True
4,Segment heterogeneity should matter,8,True
5,At least one targeted opportunity should be id...,[Female Gamer],True
6,Decision-ready broad rollout language should n...,False,True
7,Human review should be required for negative o...,True,True


Case 1 benchmark pass rate: 100.00%


### 5.6.1 Case 1 Interpretation

ClaimCheck recovered the expected benchmark pattern for the GameFun treatment-control case.

The system correctly identified that treatment and control were balanced, the campaign produced statistically significant purchase lift, and the overall rollout economics were negative under the provided value and cost assumptions. It also identified financially positive segment opportunity, which supports targeted review rather than broad rollout.

The important product result is that ClaimCheck separated three ideas that are often conflated in business reporting:

1. **Statistical lift:** the campaign increased purchase probability.
2. **Business value:** broad rollout is not financially attractive under the current economics.
3. **Decision governance:** causal language is allowed, but broad rollout recommendation language is not decision-ready and requires human review.

This is the core value of ClaimCheck: it prevents a statistically true finding from becoming an unsupported executive recommendation.

## 8. What moves into CrewAI tomorrow

This notebook deliberately proves the analytical spine before adding orchestration. The next step is to wrap the same tools in a CrewAI Python workflow.

**Recommended crew:**
- **Claim Framer** — LLM agent; turns messy stakeholder language into structured hypothesis JSON.
- **Method Scout** — LLM + rules agent; reads the data profile and selects the appropriate method module from the registry.
- **Validity Guard** — LLM agent; explains risks, assigns exception status, and enforces whether the case can auto-clear.
- **Brief Builder** — LLM agent; writes the final executive-facing brief within the language allowed by the evidence tier.

**Tools called by the crew:**
- **Data Profiler** — deterministic Python.
- **Stats Engine** — deterministic Python.
- **Business Impact Calculator** — deterministic Python.
- **Tier-Locked Language Guardrail** — deterministic Python.
- **Audit Log Generator** — deterministic Python.

**Why this split matters:** agents interpret and route; tools calculate and log. This is how ClaimCheck avoids hallucinated statistics while still behaving like an autonomous analytical reviewer.
